# 한국어 핵심 요약: 데이터 준비 · 모델 학습 · 평가

위에서부터 전체 실행하면 데이터 준비와 필요한 학습을 자동으로 진행함
실행 스위치는 없음

- 저장 모델의 설정·입력·학습 코드가 같으면 해당 학습을 건너뜀
- 하이퍼파라미터, 데이터, 토크나이저, 시작 모델, 학습 코드가 바뀌면 해당 단계부터 새로 학습함
- 같은 설정으로 다시 실행하면 새로 학습한 모델도 재사용함
- 새 결과는 `artifacts/training_runs/`에 설정별로 저장하므로 기존 서비스 모델과 기록을 보존함

첫 부분의 설정을 수정한 뒤 커널을 다시 시작하고 전체 실행할 것. 기본 장치는 로컬 CPU임
학습 뒤 검증과 개발 비교를 실행함 별도 최종 test 기록은 고정 모델의 기존 평가로 구분함


### 환경과 실행 단계 선택

로컬 환경을 준비합니다. 전체 실행 시 저장 모델과 현재 설정을 비교해 학습 여부를 자동 결정합니다.

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime
import gc
import hashlib
from io import BytesIO
from itertools import combinations
import json
import math
import os
from pathlib import Path
import random
import re
import time
import unicodedata
import zipfile
from zoneinfo import ZoneInfo
import copy
import types
import sys
import inspect
from importlib.metadata import version

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MLFLOW_ENABLE_TELEMETRY"] = "false"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from torch import nn
from transformers import AutoConfig, BartForConditionalGeneration, DataCollatorForSeq2Seq
from transformers import PreTrainedTokenizerFast, get_linear_schedule_with_warmup
from transformers.modeling_outputs import BaseModelOutput
from safetensors.torch import load_file, save_file
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "requirements-nlp.txt").is_file())
DATA_DIR = ROOT / "data/aihub_summary"
DEVICE = "cpu"
torch.set_num_threads(min(4, os.cpu_count() or 1))


print("프로젝트:", ROOT)
print("연산 장치:", DEVICE, "/ PyTorch:", torch.__version__)


### 결과 파일 읽기와 저장

학습 기록은 JSON으로 읽고 임시 파일을 완성한 뒤 저장합니다.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


### 학습 설정

학습률·epoch·배치·시드·학습할 층을 여기서 변경합니다. 변경한 설정은 학습과 재사용 판단 양쪽에 동일하게 적용됩니다.

In [ ]:
# 값을 바꾸고 전체 실행하면 해당 단계와 영향을 받는 후속 단계가 다시 학습됩니다.
BASE_SETTINGS = {
    "pilot": {"seed": 20261103, "train_size": 256, "validation_size": 64,
              "monitor_size": 64, "epochs": 1, "batch_size": 2,
              "gradient_accumulation": 4, "learning_rate": 3e-5,
              "weight_decay": .01, "max_grad_norm": 1., "warmup_ratio": .06,
              "validation_every_updates": 8, "gradient_checkpointing": False},
    "domain": {"seed": 20261104, "train_size": 768, "validation_size": 96,
               "monitor_size": 64, "epochs": 1, "batch_size": 1,
               "gradient_accumulation": 16, "learning_rate": 3e-5,
               "weight_decay": .01, "max_grad_norm": 1., "warmup_ratio": .06,
               "validation_every_updates": 8, "gradient_checkpointing": True},
}
BASE_GENERATION = {"max_new_tokens": 64, "min_new_tokens": 5, "num_beams": 2,
                   "do_sample": False, "no_repeat_ngram_size": 3, "length_penalty": 1.}
SELECTOR_SETTINGS = {"seed": 20261105, "train_count": 256, "tuning_count": 32,
                     "epochs": 60, "batch_size": 64, "learning_rate": 3e-4,
                     "weight_decay": .01, "hidden_units": 96, "dropout": .1,
                     "label_minimum": .08, "label_relative": .7}
GENERATOR_SETTINGS = {"seed": 20261105, "train_count": 256, "tuning_count": 24,
                      "epochs": 1, "learning_rate": 2e-5, "weight_decay": .01,
                      "gradient_accumulation": 8, "max_grad_norm": 1.,
                      "trainable_layers": [4, 5], "minimum_novel_ratio": .05,
                      "max_target_tokens": 96, "validation_every_samples": 64}
FACTUAL_SETTINGS = {"seed": 20261104, "learning_rates": [1e-5, 2e-5, 5e-5],
                    "epochs": 3, "gradient_accumulation": 8, "weight_decay": .01,
                    "max_grad_norm": 1., "repeat_penalty_weight": .1,
                    "trainable_layers": [4, 5], "tuning_count": 32,
                    "development_count": 32, "generation_tuning_count": 16}
TOKEN_SETTINGS = {"source_length_multiple": 32, "target_length_multiple": 8}


### 저장 모델과 현재 실행 조건 비교

설정, 데이터·시작 가중치의 식별값, 실제 학습 함수, 라이브러리 버전을 비교합니다. 관련 없는 그래프·설명 수정은 재학습을 일으키지 않습니다.

In [ ]:
CACHE_DIRECTORY = ROOT / "artifacts/training_cache"
TRAINING_DIRECTORY = ROOT / "artifacts/training_runs"
FILE_HASH_PATH = CACHE_DIRECTORY / "file_hashes.json"
FILE_HASHES = read_json(FILE_HASH_PATH) if FILE_HASH_PATH.exists() else {}
TRAINING_STEPS = {}


def json_fingerprint(value):
    encoded = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


def file_fingerprint(path):
    path = Path(path).resolve()
    stat = path.stat()
    key = str(path)
    stamp = [stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns]
    previous = FILE_HASHES.get(key, {})
    if previous.get("stamp") != stamp:
        with path.open("rb") as stream:
            digest = hashlib.file_digest(stream, "sha256").hexdigest()
        FILE_HASHES[key] = {"stamp": stamp, "sha256": digest}
    return FILE_HASHES[key]["sha256"]


def code_fingerprint(function):
    # 파일명·셀 번호·주석·공백을 제외하고 실제 실행 코드와 기본 인수를 비교합니다.
    function = inspect.unwrap(function)
    def describe(value):
        if isinstance(value, types.CodeType):
            return {"bytecode": value.co_code.hex(), "names": value.co_names,
                    "variables": value.co_varnames, "constants": [describe(v) for v in value.co_consts],
                    "arguments": [value.co_argcount, value.co_kwonlyargcount, value.co_posonlyargcount],
                    "flags": value.co_flags, "free_variables": value.co_freevars,
                    "cell_variables": value.co_cellvars, "exceptions": value.co_exceptiontable.hex()}
        if isinstance(value, (tuple, list)):
            return [describe(v) for v in value]
        if isinstance(value, (set, frozenset)):
            return sorted((describe(v) for v in value), key=lambda item: json.dumps(item, sort_keys=True, ensure_ascii=False))
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return repr(value)
    return json_fingerprint({"code": describe(function.__code__),
                             "defaults": describe(function.__defaults__),
                             "keyword_defaults": function.__kwdefaults__})


def cache_context(name, settings, inputs, functions, files):
    recipe = {"settings": copy.deepcopy(settings), "inputs": copy.deepcopy(inputs),
              "code": {function.__name__: code_fingerprint(function) for function in functions},
              "runtime": {"python": list(sys.version_info[:2]), "device": DEVICE,
                          "dtype": str(torch.get_default_dtype()), "threads": torch.get_num_threads(),
                          "matmul_precision": torch.get_float32_matmul_precision(),
                          **{package: version(package) for package in ["torch", "transformers", "numpy", "safetensors"]}}}
    fingerprint = json_fingerprint(recipe)
    record_path = CACHE_DIRECTORY / name / f"{fingerprint}.json"
    directory = TRAINING_DIRECTORY / name / fingerprint[:20]
    context = {"name": name, "fingerprint": fingerprint, "recipe": recipe,
               "record_path": record_path, "directory": directory,
               "paths": {key: directory / filename for key, filename in files.items()}, "cached": False}
    if record_path.exists():
        try:
            saved = read_json(record_path)
            paths = {key: ROOT / value for key, value in saved["files"].items()}
            valid = saved.get("status") == "complete" and saved.get("recipe") == recipe and paths.keys() == files.keys()
            if valid:
                valid = all(path.is_file() and file_fingerprint(path) == saved["sha256"][key]
                            for key, path in paths.items())
        except (OSError, ValueError, TypeError, KeyError, AttributeError):
            valid = False
        if valid:
            context.update(paths=paths, directory=ROOT / saved["directory"], cached=True)
    TRAINING_STEPS[name] = context
    return context


def save_training_cache(context):
    assert all(path.is_file() for path in context["paths"].values()), "학습 결과 파일이 빠졌습니다."
    write_json(context["record_path"], {
        "status": "complete", "fingerprint": context["fingerprint"], "recipe": context["recipe"],
        "directory": context["directory"].relative_to(ROOT).as_posix(),
        "files": {key: path.relative_to(ROOT).as_posix() for key, path in context["paths"].items()},
        "sha256": {key: file_fingerprint(path) for key, path in context["paths"].items()},
    })
    write_json(FILE_HASH_PATH, FILE_HASHES)


def finish_step(context, build):
    if context["cached"]:
        print(f"[학습 생략] {context['name']}: 설정과 저장 결과가 같습니다.")
    else:
        print(f"[새 실행] {context['name']}: 모델이 없거나 설정·입력·코드가 변경되었습니다.", flush=True)
        context["directory"].mkdir(parents=True, exist_ok=True)
        build()
        save_training_cache(context)
    return context


## 1. 공통 입력 · 생성 · 평가 함수

### 기본 모델 경로

이미 확보한 로컬 모델과 서비스 선택 기록을 사용합니다.

In [ ]:
MODEL_RELEASE = ROOT / "artifacts/model_release.json"
TOKENIZER_DIR = ROOT / "artifacts/kobart_base_v2"


### 텍스트 정규화와 파일 식별값

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFC", text)
    text = " ".join(text.split())
    return text.replace("⁓", "~").replace("⁚", ":")


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_release():
    return dict(ACTIVE_BASE_RELEASE)


### 로컬 토크나이저와 KoBART 불러오기

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def load_tokenizer():
    import os
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    from transformers import PreTrainedTokenizerFast
    return PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR, local_files_only=True)


def input_ids(text, tokenizer):
    normalized = normalize_text(text)
    ids = tokenizer.encode(normalized, add_special_tokens=False)
    return [tokenizer.bos_token_id, *ids, tokenizer.eos_token_id]


def load_model(model_directory, expected_sha256=None):
    import torch
    from transformers import BartForConditionalGeneration
    directory = Path(model_directory).resolve()
    assert directory.is_relative_to(ROOT / "artifacts"), "프로젝트에서 확인한 모델만 불러옵니다."
    if expected_sha256:
        assert sha256(directory / "model.safetensors") == expected_sha256, "모델 파일 식별값이 다릅니다."
    torch.set_num_threads(4)
    model = BartForConditionalGeneration.from_pretrained(directory, local_files_only=True,
                use_safetensors=True, attn_implementation="sdpa")
    model.eval()
    return model


### 요약 생성과 숫자·반복 확인

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def generate(text, model, tokenizer, settings, max_source_tokens=576):
    import torch
    normalized = normalize_text(text)
    if not normalized:
        raise ValueError("요약할 원문을 입력해 주세요.")
    ids = input_ids(normalized, tokenizer)
    if len(ids) > max_source_tokens:
        raise ValueError(f"현재 원문은 {len(ids)}토큰입니다. {max_source_tokens}토큰 이하의 문단을 입력해 주세요. 원문을 자동으로 자르지 않습니다.")
    start = time.perf_counter()
    data = torch.tensor([ids], dtype=torch.long, device=next(model.parameters()).device)
    with torch.inference_mode():
        output = model.generate(input_ids=data, attention_mask=torch.ones_like(data),
                                use_cache=True, **settings)
    summary = tokenizer.decode(output[0], skip_special_tokens=True,
                               clean_up_tokenization_spaces=False).strip()
    return {"source": normalized, "summary": summary, "source_tokens": len(ids),
            "seconds": time.perf_counter()-start, **review_signals(normalized, summary)}


def number_tokens(text):
    return {number.replace(",", "") for number in re.findall(r"[0-9]+(?:[.,][0-9]+)*", text)}


def char_lcs_f1(prediction, reference):
    a, b = re.sub(r"\s+", "", prediction), re.sub(r"\s+", "", reference)
    if not a or not b:
        return 0.0
    previous = [0] * (len(b)+1)
    for char in a:
        current = [0]
        for j, other in enumerate(b, 1):
            current.append(previous[j-1]+1 if char == other else max(previous[j], current[-1]))
        previous = current
    return 2*previous[-1]/(len(a)+len(b))


def sentence_candidates(source):
    # 소수점 내부를 문장 끝으로 나누지 않습니다.
    return [s.strip() for s in re.split(r"(?<=[.!?。])\s+", normalize_text(source)) if s.strip()]


def review_signals(source, summary):
    words = summary.split()
    triples = [tuple(words[i:i+3]) for i in range(max(0, len(words)-2))]
    related = sorted(enumerate(sentence_candidates(source)),
                     key=lambda pair: (-char_lcs_f1(summary, pair[1]), pair[0]))[:2]
    return {"numbers_not_in_source": sorted(number_tokens(summary)-number_tokens(source)),
            "source_number_tokens": sorted(number_tokens(source)),
            "repeated_word_trigrams": len(triples)-len(set(triples)),
            "related_source_sentences": [text for _, text in related]}


### 정답 요약과 ROUGE 비교

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def rouge_scores(prediction, references):
    """한글을 유지한 공백 어절 ROUGE F1. 형태소 분석 없이 직접 계산합니다."""
    def tokens(text):
        return re.findall(r"[\w]+", unicodedata.normalize("NFC", text).casefold(), flags=re.UNICODE)

    def score_pair(reference):
        p, r = tokens(prediction), tokens(reference)
        scores = {}
        for n in (1, 2):
            a = Counter(tuple(p[i:i+n]) for i in range(max(0, len(p)-n+1)))
            b = Counter(tuple(r[i:i+n]) for i in range(max(0, len(r)-n+1)))
            denominator = sum(a.values())+sum(b.values())
            scores[f"rouge{n}"] = 2*sum((a & b).values())/denominator if denominator else 0.0
        previous = [0]*(len(r)+1)
        for word in p:
            current = [0]
            for j, other in enumerate(r, 1):
                current.append(previous[j-1]+1 if word == other else max(previous[j], current[-1]))
            previous = current
        scores["rougeL"] = 2*previous[-1]/(len(p)+len(r)) if p and r else 0.0
        scores["char_lcs_f1"] = char_lcs_f1(prediction, reference)
        return scores

    pairs = [score_pair(ref) for ref in references]
    return {key: max(pair[key] for pair in pairs) for key in ("rouge1", "rouge2", "rougeL", "char_lcs_f1")}


## 2. 데이터 확인과 전처리

### 데이터 출처와 예시 확인

AI Hub 원본 경로와 이미 전처리된 학습 자료 한 건을 확인합니다. 최종 test 원문은 열지 않습니다.

In [ ]:
source_info = read_json(DATA_DIR / "source_info.json")
print("출처:", source_info["source_url"])
display(pd.DataFrame(source_info["full_archives"])[["name", "split", "json_files"]])
train_path = DATA_DIR / "news_briefing_train.jsonl"
if train_path.exists():
    with train_path.open(encoding="utf-8") as stream:
        sample = json.loads(next(stream))
    print("학습 원문:", sample["source"])
    print("정답 요약:", sample["target"])


### 데이터 분할 설정

뉴스와 보도자료를 연결 문서 그룹 단위로 train/validation/test에 나눕니다.

In [ ]:
SCOPE = {"news_r", "briefing"}
DATA_SEED = 20261103
SPLITS = ["train", "validation", "test"]
DATA_OUTPUTS = {split: DATA_DIR / f"news_briefing_{split}.jsonl" for split in SPLITS}
PREPROCESS_REPORT = DATA_DIR / "news_briefing_preprocess_report.json"


### 원본 ZIP 읽기

전처리에 사용하는 함수입니다. 실행 여부는 입력 파일과 설정을 비교해 자동 결정합니다.

In [ ]:
def clean_text(value):
    """한글 유니코드 결합 형태와 공백만 통일합니다."""
    if not isinstance(value, str):
        return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", value)).strip()

def digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


### 원문·정답 정리와 품질 표시

전처리에 사용하는 함수입니다. 실행 여부는 입력 파일과 설정을 비교해 자동 결정합니다.

In [ ]:
DOMAIN_TERMS = {
    "it": ["인공지능", "빅데이터", "클라우드", "소프트웨어", "정보통신", "사물인터넷",
           "블록체인", "스마트폰", "핀테크", "반도체", "디지털", "플랫폼", "애플리케이션"],
    "marketing": ["마케팅", "광고", "브랜드", "소비자", "고객", "캠페인", "매출",
                  "커머스", "유통", "구독", "프로모션"],
}

def document_key(row):
    # 같은 제목은 삭제 근거가 아니라, 같은 집합에 배치하기 위한 보수적 연결 기준입니다.
    title = re.sub(r"\.(?:hwp|hwpx|pdf|docx?|txt)$", "", row["title"], flags=re.I).strip()
    if not title:
        return ""
    identity = [row["doc_type"], title, row["publisher"], row["published_year"]]
    return digest(json.dumps(identity, ensure_ascii=False))


### 연결 문서 묶기와 중복 원문 통합

전처리에 사용하는 함수입니다. 실행 여부는 입력 파일과 설정을 비교해 자동 결정합니다.

In [ ]:
def connected_group_ids(rows):
    """공통 식별자·본문·제목의 연결을 전이적으로 합칩니다(Union-Find)."""
    parent = list(range(len(rows)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    first_seen = {}
    for i, row in enumerate(rows):
        for field in ["doc_id", "passage_id", "source_hash", "document_key"]:
            if not row[field]:
                continue
            token = (field, row[field])
            if token in first_seen:
                a, b = find(i), find(first_seen[token])
                parent[a] = b
            else:
                first_seen[token] = i
    members = defaultdict(list)
    for i, row in enumerate(rows):
        members[find(i)].append(row["record_id"])
    # 안정된 최소 원본 경로를 사용해 읽기 순서에 영향받지 않는 그룹 ID를 만듭니다.
    names = {root: "g_" + digest(min(ids))[:24] for root, ids in members.items()}
    return {row["record_id"]: names[find(i)] for i, row in enumerate(rows)}


### 문서 그룹으로 학습·검증·최종 평가 분리

전처리에 사용하는 함수입니다. 실행 여부는 입력 파일과 설정을 비교해 자동 결정합니다.

In [ ]:
def assign_splits(rows, seed):
    groups = defaultdict(list)
    for row in rows:
        groups[row["group_id"]].append(row)
    strata = defaultdict(list)
    for group_id, members in groups.items():
        stratum = tuple(sorted({kind for row in members for kind in row["doc_types"]}))
        strata[stratum].append(group_id)
    assignments = {}
    for stratum, group_ids in sorted(strata.items()):
        group_ids = sorted(group_ids)
        rng = random.Random(digest(str(seed) + "|" + "|".join(stratum)))
        rng.shuffle(group_ids)
        n = len(group_ids)
        n_valid = n_test = max(1, int(n * 0.1)) if n >= 10 else 0
        n_train = n - n_valid - n_test
        for i, group_id in enumerate(group_ids):
            assignments[group_id] = ("train" if i < n_train else
                                     "validation" if i < n_train + n_valid else "test")
    return assignments


### 분할 사이 정보 겹침 확인

전처리에 사용하는 함수입니다. 실행 여부는 입력 파일과 설정을 비교해 자동 결정합니다.

In [ ]:
def separation_checks(rows):
    checks = []
    for left, right in combinations(SPLITS, 2):
        a = [r for r in rows if r["split"] == left]
        b = [r for r in rows if r["split"] == right]
        for field in ["group_id", "source_hash", "doc_ids", "passage_ids", "document_keys"]:
            def values(subset):
                return {v for row in subset for v in
                        (row[field] if isinstance(row[field], list) else [row[field]]) if v}
            count = len(values(a) & values(b))
            checks.append({"left": left, "right": right, "field": field, "overlap": count})
            assert count == 0, f"{left}/{right}: {field} 겹침 {count}건"
    return checks


### 전처리 결과 저장과 재확인

전처리에 사용하는 함수입니다. 실행 여부는 입력 파일과 설정을 비교해 자동 결정합니다.

In [ ]:
def original_overlap(field, clean_rows):
    return len({r[field] for r in clean_rows if r["original_split"] == "Training" and r[field]} &
               {r[field] for r in clean_rows if r["original_split"] == "Validation" and r[field]})


### 전처리 전체 실행 함수

앞에서 정의한 정리·그룹 분할·저장 작업을 연결합니다. 입력이 바뀐 경우 자동 실행부에서 호출합니다.

In [ ]:
def prepare_preprocessing():
    raw_rows, archive_audit, all_types = [], [], Counter()

    for spec in sorted(source_info["full_archives"], key=lambda s: s["name"]):
            archive_path = DATA_DIR / spec["name"]
            with archive_path.open("rb") as handle:
                actual_sha = hashlib.file_digest(handle, "sha256").hexdigest()
            assert actual_sha == spec["sha256"], f"{spec['name']}: 원본 식별값 불일치"
            json_count = selected_count = 0
            with zipfile.ZipFile(archive_path) as archive:
                for item in sorted(archive.infolist(), key=lambda i: i.filename):
                    if item.is_dir() or not item.filename.lower().endswith(".json"):
                        continue
                    obj = json.loads(archive.read(item).decode("utf-8-sig"))
                    assert isinstance(obj, dict), item.filename
                    assert all(key in obj for key in ["Meta(Acqusition)", "Meta(Refine)", "Annotation"]), item.filename
                    meta, body, annotation = obj["Meta(Acqusition)"], obj["Meta(Refine)"], obj["Annotation"]
                    kind = meta.get("doc_type")
                    all_types[kind or "(누락)"] += 1
                    json_count += 1
                    if kind not in SCOPE:
                        continue
                    selected_count += 1
                    raw_rows.append({
                        "record_id": spec["name"] + ":" + item.filename,
                        "archive": spec["name"], "source_member": item.filename,
                        "original_split": spec["split"], "doc_type": kind,
                        "doc_id": meta.get("doc_id"), "passage_id": body.get("passage_id"),
                        "title": meta.get("doc_name"), "publisher": meta.get("publisher"),
                        "published_year": meta.get("publisher_year", meta.get("published_year")),
                        "source": body.get("passage"), "target": annotation.get("summary1"),
                    })
            assert json_count == spec["json_files"], f"{spec['name']}: 확보 당시 JSON 건수와 다름"
            archive_audit.append({"name": spec["name"], "sha256": actual_sha,
                                  "json_files": json_count, "selected_records": selected_count})
            print(f"{spec['name']}: 전체 {json_count:,}건 중 {selected_count:,}건 선택")

    assert raw_rows, "선택된 자료가 없습니다."

    assert len({r['record_id'] for r in raw_rows}) == len(raw_rows)

    display(pd.crosstab(pd.Series([r['original_split'] for r in raw_rows], name='원래 구분'),
                            pd.Series([r['doc_type'] for r in raw_rows], name='문서 유형')))

    clean_rows, excluded_records = [], []

    changed_source_count = changed_target_count = 0

    for raw in raw_rows:
            row = dict(raw)
            for key in ["source", "target", "title", "publisher", "published_year", "doc_id", "passage_id"]:
                row[key] = clean_text(raw[key])
            missing = [key for key in ["source", "target", "doc_id", "passage_id"] if not row[key]]
            if missing:
                excluded_records.append({"record_id": row["record_id"], "reason": "empty:" + ",".join(missing)})
                continue
            # 공백과 NFC 외의 문자가 사라지거나 추가되지 않았는지 모든 원문·정답에서 확인합니다.
            for key in ["source", "target"]:
                assert re.sub(r"\s", "", unicodedata.normalize("NFC", raw[key])) == re.sub(r"\s", "", row[key])
            changed_source_count += row["source"] != raw["source"]
            changed_target_count += row["target"] != raw["target"]
            row["source_hash"] = digest(row["source"])
            row["document_key"] = document_key(row)
            search_text = row["title"] + " " + row["source"]
            row["domain_candidates"] = sorted(domain for domain, terms in DOMAIN_TERMS.items()
                                              if any(term in search_text for term in terms))
            row["quality_flags"] = []
            if len(row["target"]) >= len(row["source"]):
                row["quality_flags"].append("target_not_shorter")
            if not re.search("[가-힣]", row["source"]) or not re.search("[가-힣]", row["target"]):
                row["quality_flags"].append("hangul_missing")
            clean_rows.append(row)

    assert clean_rows

    print(f"선택 {len(raw_rows):,}건 / 필수 항목 누락 제외 {len(excluded_records):,}건")

    print(f"공백·NFC 변경: 원문 {changed_source_count:,}건, 정답 {changed_target_count:,}건")

    print("숫자와 문장부호 등 나머지 문자 보존 확인 완료")

    _fixture = [
            dict(record_id="a", doc_id="d1", passage_id="p1", source_hash="x", document_key="t1"),
            dict(record_id="b", doc_id="d2", passage_id="p2", source_hash="x", document_key="t2"),
            dict(record_id="c", doc_id="d2", passage_id="p3", source_hash="y", document_key="t3"),
            dict(record_id="d", doc_id="d3", passage_id="p4", source_hash="z", document_key="t3"),
            dict(record_id="e", doc_id="d4", passage_id="p5", source_hash="w", document_key=""),
        ]

    _fg = connected_group_ids(_fixture)

    assert len({_fg[x] for x in "abcd"}) == 1 and _fg["e"] != _fg["a"]

    assert _fg == connected_group_ids(list(reversed(_fixture)))

    assert clean_text("매출  3.5억 원,\n2026-11-03, +12.5%") == "매출 3.5억 원, 2026-11-03, +12.5%"

    del _fixture, _fg

    record_groups = connected_group_ids(clean_rows)

    source_buckets = defaultdict(list)

    for row in clean_rows:
            source_buckets[row["source_hash"]].append(row)

    prepared = []

    for source_hash, bucket in sorted(source_buckets.items()):
            bucket = sorted(bucket, key=lambda r: (r["original_split"], r["record_id"]))
            representative = bucket[0]
            assert len({r["source"] for r in bucket}) == 1, "원문 해시 충돌"
            assert len({record_groups[r["record_id"]] for r in bucket}) == 1
            row = {key: representative[key] for key in
                   ["source", "target", "source_hash", "doc_type", "title", "publisher", "published_year"]}
            row.update({
                "id": "pair_" + source_hash[:24], "group_id": record_groups[representative["record_id"]],
                "doc_types": sorted({r["doc_type"] for r in bucket}),
                "doc_ids": sorted({r["doc_id"] for r in bucket}),
                "passage_ids": sorted({r["passage_id"] for r in bucket}),
                "document_keys": sorted({r["document_key"] for r in bucket if r["document_key"]}),
                "original_splits": sorted({r["original_split"] for r in bucket}),
                "reference_summaries": list(dict.fromkeys(r["target"] for r in bucket)),
                "domain_candidates": sorted({v for r in bucket for v in r["domain_candidates"]}),
                "quality_flags": sorted({v for r in bucket for v in r["quality_flags"]}),
                "source_records": [{key: r[key] for key in ["archive", "source_member", "doc_id", "passage_id"]}
                                   for r in bucket],
            })
            prepared.append(row)

    duplicate_extra_rows = len(clean_rows) - len(prepared)

    multiple_reference_pairs = sum(len(r["reference_summaries"]) > 1 for r in prepared)

    print(f"원문 {len(clean_rows):,}건 → 고유 원문 {len(prepared):,}건")

    print(f"중복 추가 행 {duplicate_extra_rows:,}건 합침 / 복수 정답 원문 {multiple_reference_pairs:,}건 보존")

    print("연결 그룹:", len(set(record_groups.values())))

    assignments = assign_splits(prepared, DATA_SEED)

    assert assignments == assign_splits(list(reversed(prepared)), DATA_SEED)

    for row in prepared:
            row["split"] = assignments[row["group_id"]]

    prepared.sort(key=lambda r: r["id"])

    assert len({r["id"] for r in prepared}) == len(prepared)

    summary_rows = []

    for split in SPLITS:
            subset = [r for r in prepared if r["split"] == split]
            summary_rows.append({"구분": split, "원문·정답 쌍": len(subset),
                                 "비율(%)": round(100 * len(subset) / len(prepared), 2),
                                 "그룹 수": len({r["group_id"] for r in subset}),
                                 "뉴스": sum(r["doc_type"] == "news_r" for r in subset),
                                 "보도자료": sum(r["doc_type"] == "briefing" for r in subset),
                                 "IT 후보": sum("it" in r["domain_candidates"] for r in subset),
                                 "마케팅 후보": sum("marketing" in r["domain_candidates"] for r in subset)})

    display(pd.DataFrame(summary_rows))

    print("IT 후보와 마케팅 후보는 서로 중복될 수 있고, 사람의 검토가 필요한 키워드 후보입니다.")

    checks = separation_checks(prepared)

    assert all(any(r["split"] == s for r in prepared) for s in SPLITS)

    assert len({r["source_hash"] for r in prepared}) == len(prepared)

    assert sum(len(r["source_records"]) for r in prepared) == len(clean_rows)

    assert all(r["source"] and r["target"] and r["target"] in r["reference_summaries"] for r in prepared)

    display(pd.DataFrame(checks))

    lengths = pd.DataFrame({"source_chars": [len(r['source']) for r in prepared],
                                "target_chars": [len(r['target']) for r in prepared]})

    display(lengths.describe(percentiles=[.5, .9, .95]).round(1))

    print("길이는 문자 수입니다. 토큰 수 확인과 길이 제한은 다음 모델 단계에서 결정합니다.")

    print("남긴 품질 확인 표시:", dict(Counter(flag for r in prepared for flag in r["quality_flags"])))

    output_records = {}

    for split, path in DATA_OUTPUTS.items():
            rows = [r for r in prepared if r["split"] == split]
            with path.open("w", encoding="utf-8", newline="\n") as handle:
                for row in rows:
                    handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
            with path.open("rb") as handle:
                sha = hashlib.file_digest(handle, "sha256").hexdigest()
            output_records[split] = {"filename": path.name, "records": len(rows),
                                     "bytes": path.stat().st_size, "sha256": sha}

    reloaded = []

    for split, path in DATA_OUTPUTS.items():
            with path.open(encoding="utf-8") as handle:
                rows = [json.loads(line) for line in handle if line.strip()]
            assert len(rows) == output_records[split]["records"]
            assert all(r["split"] == split and digest(r["source"]) == r["source_hash"] for r in rows)
            reloaded.extend(rows)

    assert len(reloaded) == len(prepared)

    assert len({r["id"] for r in reloaded}) == len(prepared)

    assert {r["id"]: r for r in reloaded} == {r["id"]: r for r in prepared}

    assert separation_checks(reloaded) == checks

    report = {
            "created_at": datetime.now(ZoneInfo("Asia/Seoul")).isoformat(timespec="seconds"),
            "pipeline_version": 1, "seed": DATA_SEED,
            "scope": {"doc_types": sorted(SCOPE), "unit": "passage_to_one_sentence_summary1",
                      "domain_filter": "none; keyword candidates only; not official labels",
                      "sample_zip_included": False},
            "source_url": source_info["source_url"], "attribution": source_info["attribution"],
            "usage_notes": source_info["usage_notes"], "inputs": archive_audit,
            "selected_records": len(raw_rows), "invalid_records_excluded": len(excluded_records),
            "excluded_records": excluded_records,
            "normalized_source_records": changed_source_count, "normalized_target_records": changed_target_count,
            "duplicate_extra_rows_merged": duplicate_extra_rows,
            "duplicate_source_groups": sum(len(v) > 1 for v in source_buckets.values()),
            "multiple_reference_pairs": multiple_reference_pairs,
            "unique_source_pairs": len(prepared), "groups": len(assignments),
            "largest_group_pairs": max(Counter(r["group_id"] for r in prepared).values()),
            "original_split_overlap": {field: original_overlap(field, clean_rows) for field in
                                       ["doc_id", "passage_id", "source_hash", "document_key"]},
            "split_policy": "custom 80/10/10 by connected group, stratified by document types; original split retained as provenance",
            "group_fields": ["doc_id", "passage_id", "source_hash", "document_key"],
            "normalization": "NFC and whitespace collapse only; no truncation, stopword, punctuation or number removal",
            "representative_target_policy": "first by original_split, record_id; all distinct normalized references preserved",
            "domain_candidate_terms": DOMAIN_TERMS,
            "split_summary": summary_rows, "separation_checks": checks, "outputs": output_records,
            "verification": {"normalization_preserves_non_whitespace_characters": True,
                             "transitive_group_fixture": True, "row_order_invariant_split": True,
                             "output_roundtrip_equal": True, "cross_split_overlap_zero": True},
            "limitations": ["No near-duplicate semantic matching", "Title grouping is conservative, not proof of identical documents",
                            "Keyword candidates need human domain review", "Reference factuality is not automatically guaranteed",
                            "Tokenization and model-specific length limits deferred to the next stage"],
        }

    PREPROCESS_REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    for split, result in output_records.items():
            print(f"{split}: {result['records']:,}쌍 저장 및 다시 읽기 검증 완료")

    print("원문·문서 ID·문단 ID·제목 키·연결 그룹의 집합 간 겹침 모두 0")

    print("점검 기록:", PREPROCESS_REPORT)


## 3. 토큰화와 학습 입력 준비

### 한 분할의 문장을 토큰 배열로 변환

원문은 BOS+본문+EOS, 정답은 요약+EOS로 만듭니다. 길이별 패딩은 학습 배치에서만 적용합니다.

In [ ]:
def tokenize_records(records, tokenizer):
    inputs, labels = [], []
    input_offsets, label_offsets = [0], [0]
    for row in records:
        source = input_ids(row["source"], tokenizer)
        target = tokenizer.encode(normalize_text(row["target"]), add_special_tokens=False) + [tokenizer.eos_token_id]
        inputs.extend(source)
        labels.extend(target)
        input_offsets.append(len(inputs))
        label_offsets.append(len(labels))
    return {
        "ids": np.asarray([row["id"] for row in records]),
        "input_ids": np.asarray(inputs, dtype=np.int32),
        "label_ids": np.asarray(labels, dtype=np.int32),
        "input_offsets": np.asarray(input_offsets, dtype=np.int64),
        "label_offsets": np.asarray(label_offsets, dtype=np.int64),
    }


def read_training_rows(split):
    path = DATA_OUTPUTS[split]
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


### 토큰 배열과 준비 기록 저장

원문과 정답을 토큰으로 변환해 저장합니다. 데이터·토크나이저·토큰화 설정이 같으면 저장 배열을 재사용합니다.

In [ ]:
def prepare_tokenization():
    report_path = TOKEN_REPORT_PATH
    tokenizer = load_tokenizer()
    origin = read_json(TOKENIZER_DIR / "model_origin.json")
    for name, info in origin["files"].items():
        assert sha256(TOKENIZER_DIR / name) == info["sha256"]
    preprocess = read_json(PREPROCESS_REPORT)
    records, arrays = {}, {}
    for split in ["train", "validation"]:
        assert sha256(DATA_OUTPUTS[split]) == preprocess["outputs"][split]["sha256"]
        records[split] = read_training_rows(split)
        arrays[split] = tokenize_records(records[split], tokenizer)
    max_source = math.ceil(np.diff(arrays["train"]["input_offsets"]).max() / TOKEN_SETTINGS["source_length_multiple"]) * TOKEN_SETTINGS["source_length_multiple"]
    max_target = math.ceil(np.diff(arrays["train"]["label_offsets"]).max() / TOKEN_SETTINGS["target_length_multiple"]) * TOKEN_SETTINGS["target_length_multiple"]
    config = AutoConfig.from_pretrained(TOKENIZER_DIR, local_files_only=True)
    assert max(max_source, max_target) <= config.max_position_embeddings
    caches = {}
    for split, values in arrays.items():
        assert np.diff(values["input_offsets"]).max() <= max_source
        assert np.diff(values["label_offsets"]).max() <= max_target
        assert tokenizer.unk_token_id not in values["label_ids"]
        path = TOKEN_FILES[split]
        np.savez_compressed(path, **values)
        with np.load(path, allow_pickle=False) as restored:
            assert all(np.array_equal(restored[key], value) for key, value in values.items())
        caches[split] = {"filename": path.name, "rows": len(records[split]),
                         "sha256": sha256(path), "source_jsonl_sha256": sha256(DATA_OUTPUTS[split])}
    preparation = {
        "status": "tokenization_complete", "token_caches": caches,
        "model": {"weights_sha256": origin["files"]["model.safetensors"]["sha256"]},
        "tokenizer": {"tokenizer_sha256": sha256(TOKENIZER_DIR / "tokenizer.json"),
                      "input_format": "BOS + source + EOS", "label_format": "summary + EOS"},
        "length_policy": {"max_source_length": int(max_source), "max_target_length": int(max_target)},
        "training_plan": {"seed": DATA_SEED}, "test_data_read": False,
    }
    write_json(report_path, preparation)
    return preparation


## 4. 시험 학습과 주제 선별 추가 학습

두 실험의 공통 학습·저장 코드를 한 번만 정의합니다. 두 실험의 기본값은 1 epoch이며, 학습 설정 셀에서 변경할 수 있습니다.

### 문서 유형 비율에 맞춰 그룹별 표본 선택

같은 문서 그룹은 한 번만 선택합니다. 시험 학습 256건·검증 64건에 사용합니다.

In [ ]:
def stratified_indices(records, size, seed):
    kinds = sorted({r["doc_type"] for r in records})
    counts = np.asarray([sum(r["doc_type"] == k for r in records) for k in kinds])
    exact = size * counts / counts.sum()
    quotas = np.floor(exact).astype(int)
    for i in np.argsort(-(exact - quotas))[:size - int(quotas.sum())]:
        quotas[i] += 1
    rng, chosen, groups = np.random.default_rng(seed), [], set()
    for kind, quota in zip(kinds, quotas):
        pool = sorted((i for i, r in enumerate(records) if r["doc_type"] == kind), key=lambda i: records[i]["id"])
        accepted = 0
        for i in rng.permutation(pool):
            i = int(i)
            if records[i]["group_id"] in groups:
                continue
            chosen.append(i)
            groups.add(records[i]["group_id"])
            accepted += 1
            if accepted == quota:
                break
        assert accepted == quota
    assert len(chosen) == size
    return sorted(chosen, key=lambda i: records[i]["id"])


### 실험 설정과 고정 표본 준비

첫 설정 셀의 학습 건수·시드·epoch·배치 등을 그대로 사용합니다. 주제 표본 설정이 바뀌면 표본도 다시 선택합니다.

In [ ]:
def training_config(stage):
    assert stage in BASE_SETTINGS
    settings = copy.deepcopy(BASE_SETTINGS[stage])
    for key in ["epochs", "train_size", "validation_size", "batch_size", "gradient_accumulation"]:
        assert isinstance(settings[key], int) and settings[key] > 0, key
    plan = read_json(DOMAIN_PLAN_PATH) if stage == "domain" else None
    return settings, plan


def training_samples(rows, settings, plan):
    seed = settings["seed"]
    indices = {}
    for split, size_key, offset in [("train", "train_size", 0), ("validation", "validation_size", 1)]:
        size = settings[size_key]
        planned = plan["splits"][split]["selected_rows"] if plan else []
        lookup = {row["id"]: i for i, row in enumerate(rows[split])}
        if plan and seed == plan["seed"] and size == len(planned) and all(row["id"] in lookup for row in planned):
            indices[split] = [lookup[row["id"]] for row in planned]
        elif plan:
            pool = domain_pool(rows[split], load_tokenizer(), plan["rules"])
            selected = stratified_indices(pool, size, seed + offset)
            indices[split] = [lookup[pool[i]["id"]] for i in selected]
        else:
            indices[split] = stratified_indices(rows[split], size, seed + offset)
    for field in ["group_id", "source_hash", "doc_ids", "passage_ids", "document_keys"]:
        keys = {}
        for split, values in indices.items():
            keys[split] = {value for i in values for value in
                           (rows[split][i][field] if isinstance(rows[split][i][field], list) else [rows[split][i][field]])}
        assert not keys["train"] & keys["validation"], field
    monitor = stratified_indices([rows["train"][i] for i in indices["train"]],
                                 min(settings["monitor_size"], len(indices["train"])), seed + 2)
    indices["monitor"] = [indices["train"][i] for i in monitor]
    return indices


### 배치·검증 손실·생성 결과 준비

정답 패딩 -100은 손실에서 제외합니다. 배치마다 정답 토큰 수로 가중한 검증 손실을 계산합니다.

In [ ]:
def token_feature(arrays, index, preparation):
    start, end = arrays["input_offsets"][index:index + 2]
    label_start, label_end = arrays["label_offsets"][index:index + 2]
    source = arrays["input_ids"][start:end].tolist()
    target = arrays["label_ids"][label_start:label_end].tolist()
    assert len(source) <= preparation["length_policy"]["max_source_length"]
    assert len(target) <= preparation["length_policy"]["max_target_length"]
    return {"input_ids": source, "attention_mask": [1] * len(source), "labels": target}


def token_batch(arrays, indices, collator, preparation):
    return {key: value.to(DEVICE) for key, value in
            collator([token_feature(arrays, i, preparation) for i in indices]).items()}


@torch.inference_mode()
def model_loss(model, arrays, indices, collator, preparation, batch_size):
    model.eval()
    total, tokens = 0.0, 0
    for offset in range(0, len(indices), batch_size):
        batch = token_batch(arrays, indices[offset:offset + batch_size], collator, preparation)
        count = int((batch["labels"] != -100).sum())
        loss = model(**batch, use_cache=False).loss
        assert torch.isfinite(loss)
        total += loss.item() * count
        tokens += count
    return total / tokens


def validation_predictions(model, tokenizer, records, indices, generation):
    return {records[i]["id"]: generate(records[i]["source"], model, tokenizer, generation,
                                      read_json(TOKEN_REPORT_PATH)["length_policy"]["max_source_length"])["summary"]
            for i in indices}


### 설정한 epoch 학습과 손실 기록

설정한 epoch 수만큼 학습하고 손실을 기록합니다.

In [ ]:
def fit_one_epoch(model, arrays, indices, collator, preparation, settings, output):
    def measure(update, seen):
        return {"update": update, "epoch_fraction": seen / len(indices["train"]),
                "train_monitor_loss": model_loss(model, arrays["train"], indices["monitor"], collator, preparation, settings["batch_size"]),
                "validation_loss": model_loss(model, arrays["validation"], indices["validation"], collator, preparation, settings["batch_size"])}

    history, trace = [measure(0, 0)], []
    window_size = settings["batch_size"] * settings["gradient_accumulation"]
    updates_per_epoch = math.ceil(len(indices["train"]) / window_size)
    total_updates = updates_per_epoch * settings["epochs"]
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"])
    scheduler = get_linear_schedule_with_warmup(optimizer, math.ceil(total_updates * settings["warmup_ratio"]), total_updates)
    start = time.perf_counter()
    for epoch in range(settings["epochs"]):
        order = np.random.default_rng(settings["seed"] + 3 + epoch).permutation(indices["train"]).tolist()
        for offset in range(0, len(order), window_size):
            selected = order[offset:offset + window_size]
            batches = [token_batch(arrays["train"], selected[i:i + settings["batch_size"]], collator, preparation)
                       for i in range(0, len(selected), settings["batch_size"])]
            token_count = sum(int((batch["labels"] != -100).sum()) for batch in batches)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            mean_loss = 0.0
            for batch in batches:
                loss = model(**batch, use_cache=False).loss
                assert torch.isfinite(loss)
                weighted = loss * int((batch["labels"] != -100).sum()) / token_count
                weighted.backward()
                mean_loss += weighted.item()
            gradient = torch.nn.utils.clip_grad_norm_(model.parameters(), settings["max_grad_norm"], error_if_nonfinite=True)
            learning_rate = optimizer.param_groups[0]["lr"]
            optimizer.step()
            scheduler.step()
            update = len(trace) + 1
            seen = epoch * len(order) + offset + len(selected)
            trace.append({"update": update, "samples_seen": seen, "training_mode_loss": mean_loss,
                          "learning_rate_used": learning_rate, "gradient_norm_before_clip": float(gradient)})
            if update % settings["validation_every_updates"] == 0 or (offset + len(selected)) == len(order):
                history.append(measure(update, seen))
                print(f"갱신 {update}/{total_updates} / 검증 손실 {history[-1]['validation_loss']:.4f}")
            write_json(output / "progress.json", {"status": "training", "history": history, "training_trace": trace})
    return history, trace, time.perf_counter() - start


### 시험·주제 학습 실행과 저장 모델 확인

저장 여부 판단은 자동 실행부에서 처리합니다. 새 학습 함수는 현재 설정으로 학습·저장하고, 저장 모델의 손실이 같은지 확인합니다.

In [ ]:
def train_kobart(stage, output):
    result_path = output / f"{stage}_results.json"
    preparation = read_json(TOKEN_REPORT_PATH)
    settings, plan = training_config(stage)
    torch.manual_seed(settings["seed"])
    rows = {split: read_training_rows(split) for split in ["train", "validation"]}
    arrays = {}
    for split, info in preparation["token_caches"].items():
        assert sha256(TOKEN_FILES[split]) == info["sha256"]
        assert sha256(DATA_OUTPUTS[split]) == info["source_jsonl_sha256"]
        with np.load(TOKEN_FILES[split], allow_pickle=False) as cache:
            arrays[split] = {key: cache[key] for key in cache.files}
        assert arrays[split]["ids"].tolist() == [row["id"] for row in rows[split]]
    indices = training_samples(rows, settings, plan)
    initial = BASE_DIRECTORIES["pilot"] if plan else TOKENIZER_DIR
    expected = file_fingerprint(initial / "model.safetensors")
    assert sha256(initial / "model.safetensors") == expected
    model = load_model(initial, expected).to(DEVICE)
    if settings["gradient_checkpointing"]:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    tokenizer = load_tokenizer()
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True,
                                     pad_to_multiple_of=8, label_pad_token_id=-100, return_tensors="pt")
    generation = dict(BASE_GENERATION)
    before = validation_predictions(model, tokenizer, rows["validation"], indices["validation"], generation)
    history, trace, seconds = fit_one_epoch(model, arrays, indices, collator, preparation, settings, output)
    model.eval()
    model.generation_config.update(**generation)
    model.save_pretrained(output, safe_serialization=True)
    after = validation_predictions(model, tokenizer, rows["validation"], indices["validation"], generation)
    check = token_batch(arrays["validation"], indices["validation"][:2], collator, preparation)
    with torch.inference_mode():
        expected_loss = model(**check, use_cache=False).loss.item()
    collator.model = None
    del model
    gc.collect()
    restored = load_model(output).to(DEVICE)
    with torch.inference_mode():
        restored_loss = restored(**check, use_cache=False).loss.item()
    assert math.isclose(expected_loss, restored_loss, rel_tol=1e-5, abs_tol=1e-5)
    generations = []
    for i in indices["validation"]:
        row = rows["validation"][i]
        generations.append({"id": row["id"], "source": row["source"], "target": row["target"],
                            "doc_type": row["doc_type"], "references": row["reference_summaries"],
                            "lead_sentence": sentence_candidates(row["source"])[0],
                            "before": before[row["id"]], "after": after[row["id"]]})
    selection = {"train_ids": [rows["train"][i]["id"] for i in indices["train"]],
                 "validation_ids": [rows["validation"][i]["id"] for i in indices["validation"]],
                 "monitor_train_ids": [rows["train"][i]["id"] for i in indices["monitor"]],
                 "example_ids": [rows["validation"][i]["id"] for kind in ["news_r", "briefing"]
                                 for i in [j for j in indices["validation"] if rows["validation"][j]["doc_type"] == kind][:4]]}
    saved = {"status": "complete", "settings": settings, "generation_settings": generation,
             "selection": selection, "history": history, "training_trace": trace,
             "training_seconds": seconds, "generations": generations, "device": DEVICE,
             "token_caches": preparation["token_caches"], "tokenization_policy": preparation["tokenizer"],
             "base_weights_sha256": preparation["model"]["weights_sha256"],
             "initial_weights_sha256": expected, "checkpoint_sha256": sha256(output / "model.safetensors"),
             "generation_config_sha256": sha256(output / "generation_config.json"),
             "reload_loss_difference": abs(expected_loss - restored_loss), "test_data_read": False}
    if plan:
        saved["domain_plan_sha256"] = sha256(ROOT / "artifacts/domain_training_plan.json")
    write_json(result_path, saved)
    return saved


## 5. 검증 · 모델 고정 · 별도 최종 평가

### 평가 경로

모델 선택 기준과 주제 선별 계획은 기존의 고정 파일을 사용합니다.

In [ ]:
DOMAIN_DIRECTORY = ROOT / "artifacts/kobart_domain"
DOMAIN_PLAN_PATH = ROOT / "artifacts/domain_training_plan.json"


### 방법별 평가 집계와 검증 결과 계산

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def aggregate_metrics(rows, methods):
    result = {}
    for method in methods:
        metrics = [rouge_scores(row[method], row["references"]) for row in rows]
        signals = [review_signals(row["source"], row[method]) for row in rows]
        result[method] = {key: float(np.mean([m[key] for m in metrics])) for key in metrics[0]}
        result[method].update(n=len(rows), empty_outputs=sum(not row[method] for row in rows),
                             novel_number_signals=sum(bool(s["numbers_not_in_source"]) for s in signals),
                             repetition_signals=sum(s["repeated_word_trigrams"] > 0 for s in signals))
        for row, metric in zip(rows, metrics):
            row.setdefault("evaluation_metrics", {})[method] = metric
    return result


def evaluate_validation():
    result = read_json(DOMAIN_DIRECTORY / "domain_results.json")
    assert result["status"] == "complete", "주제 선별 학습이 아직 완료되지 않았습니다."
    assert result["domain_plan_sha256"] == sha256(DOMAIN_PLAN_PATH)
    rows = result["generations"]
    metrics = aggregate_metrics(rows, ["lead_sentence", "before", "after"])
    difference = np.array([r["evaluation_metrics"]["after"]["char_lcs_f1"]-
                           r["evaluation_metrics"]["before"]["char_lcs_f1"] for r in rows])
    draws = np.random.default_rng(20261105).integers(0, len(rows), size=(2000, len(rows)))
    distribution = difference[draws].mean(axis=1)
    report = {"status": "complete", "split": "validation", "n": len(rows), "metrics": metrics,
              "char_lcs_paired_difference": float(difference.mean()),
              "char_lcs_paired_bootstrap_95_interval": np.quantile(distribution, [.025, .975]).tolist(),
              "bootstrap_repetitions": 2000, "bootstrap_seed": 20261105,
              "metric_definition": "Unicode word tokens: regex [\\w]+, NFC, casefold, no stemming; per-reference maximum F1 for each metric. Independent implementation, not a standard Korean benchmark.",
              "test_data_read": False, "model_sha256": result["checkpoint_sha256"],
              "generations": rows, "example_ids": result["selection"]["example_ids"],
              "limits": ["Curated 96-row validation subset", "Lexical overlap is not factual accuracy",
                         "Bootstrap interval describes resampling of this fixed subset, not all news"]}
    write_json(VALIDATION_REPORT_PATH, report)
    print(json.dumps({k: report[k] for k in ["n", "metrics", "char_lcs_paired_difference", "char_lcs_paired_bootstrap_95_interval"]}, ensure_ascii=False, indent=2))


### 평가 대상 주제 범위 선택

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def domain_pool(records, tokenizer, rules):
    def norm(text):
        return re.sub(r"\s+", "", unicodedata.normalize("NFKC", text)).casefold()

    def hits(text):
        normalized = norm(text)
        return {key: [term for term in terms if term in normalized] for key, terms in rules["terms"].items()}

    candidates = []
    for row in records:
        sh, th, yh = hits(row["source"]), hits(row["title"]), hits(row["target"])
        ss, ts, ys = [set(sum(values.values(), [])) for values in (sh, th, yh)]
        categories = [key for key in rules["terms"] if yh[key] and sh[key]]
        relevant = (bool(categories) and len(ss) >= rules["minimum_distinct_source_terms"] and
                    bool((ys | ts)-set(rules["weak_terms"])) and
                    any(action in norm(row["target"]) for action in rules["action_terms"]))
        if not relevant or any(term in norm(row["title"]+row["target"]) for term in rules["excluded_title_target_terms"]):
            continue
        length = len(input_ids(row["source"], tokenizer))
        target_length = len(tokenizer.encode(row["target"].replace("⁓", "~").replace("⁚", ":"), add_special_tokens=False))+1
        if (row["quality_flags"] or length > rules["max_source_tokens"] or
                not rules["min_target_tokens"] <= target_length <= rules["max_target_tokens"]):
            continue
        score = 3*len(ys)+2*len(ts)+min(len(ss), 8)
        candidates.append({**row, "categories": categories, "domain_score": score, "source_tokens": length})
    by_group = {}
    for row in sorted(candidates, key=lambda r: (-r["domain_score"], r["id"])):
        by_group.setdefault(row["group_id"], row)
    return list(by_group.values())


### 검증과 원문 대조 기록으로 서비스 모델 고정

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def select_service_model():
    """test를 읽지 않고, 사전에 고정한 기준과 원문 검토로 모델을 선택합니다."""
    result = read_json(DOMAIN_DIRECTORY / "domain_results.json")
    validation_report = read_json(DOMAIN_DIRECTORY / "validation_evaluation.json")
    review = read_json(DOMAIN_DIRECTORY / "factual_review.json")
    protocol_path = ROOT / "artifacts/evaluation_protocol.json"
    protocol = read_json(protocol_path)
    assert result["status"] == validation_report["status"] == review["status"] == "complete"
    assert validation_report["model_sha256"] == result["checkpoint_sha256"]
    assert len(review["entries"]) == 8
    assert {e["id"] for e in review["entries"]} == set(validation_report["example_ids"])
    fields = protocol["validation_review_fields"]
    counts = {}
    for stage in ["before", "after"]:
        for row in review["entries"]:
            assert set(row[stage]) == set(fields)
            assert all(v in [0, 1] for v in row[stage].values())
        counts[stage] = {"critical_error_documents": sum(any(row[stage][f] for f in fields[:3]) for row in review["entries"]),
                         "key_omission_documents": sum(row[stage]["key_omission"] for row in review["entries"]),
                         "flags_by_type": {f: sum(row[stage][f] for row in review["entries"]) for f in fields}}
    quantitative = (validation_report["metrics"]["after"]["char_lcs_f1"] >= validation_report["metrics"]["before"]["char_lcs_f1"] and
                    result["history"][-1]["validation_loss"] <= result["history"][0]["validation_loss"])
    qualitative = all(counts["after"][k] <= counts["before"][k] for k in ["critical_error_documents", "key_omission_documents"])
    chosen = "artifacts/kobart_domain" if quantitative and qualitative else "artifacts/kobart_pilot"
    weights = sha256(ROOT / chosen / "model.safetensors")
    assert weights == (result["checkpoint_sha256"] if chosen.endswith("domain") else result["initial_weights_sha256"])
    release = {"selection_locked_before_test": True, "selected_at": datetime.now().astimezone().isoformat(),
               "model_directory": chosen, "weights_sha256": weights,
               "generation_settings": result["generation_settings"], "max_source_tokens": protocol["max_source_tokens"],
               "evaluation_protocol_sha256": sha256(protocol_path), "domain_plan_sha256": sha256(DOMAIN_PLAN_PATH),
               "validation_report_sha256": sha256(DOMAIN_DIRECTORY / "validation_evaluation.json"),
               "factual_review_sha256": sha256(DOMAIN_DIRECTORY / "factual_review.json"),
               "quantitative_rule_passed": quantitative, "qualitative_rule_passed": qualitative,
               "review_counts": counts, "test_data_read_for_selection": False,
               "selection_reason": ("검증 손실·문자 LCS가 개선되고 고정 8건의 중요 오류·핵심 누락이 늘지 않아 주제 선별 모델을 선택했습니다."
                                    if quantitative and qualitative else
                                    "주제 선별 모델이 사전에 정한 검증 기준을 모두 통과하지 못해 기존 시험 학습 모델을 유지했습니다."),
               "review_limit": "8건의 AI 보조 원문 대조이며 독립적인 사람 평가가 아닙니다."}
    target = ROOT / "artifacts/model_release.json"
    if target.exists():
        previous = read_json(target)
        release["selected_at"] = previous["selected_at"]
        assert release == previous, "이미 고정한 모델과 선택 근거가 다릅니다. test 평가 뒤 변경할 수 없습니다."
        print("이미 고정된 동일 모델을 재사용합니다.")
    else:
        assert not (DOMAIN_DIRECTORY / "test_evaluation.json").exists(), "test 평가 뒤 모델을 새로 선택할 수 없습니다."
        write_json(target, release)
    print(json.dumps(release, ensure_ascii=False, indent=2))


### 고정한 모델의 별도 test 평가

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def evaluate_test():
    release = read_json(ROOT / "artifacts/model_release.json")
    protocol = read_json(ROOT / "artifacts/evaluation_protocol.json")
    assert release["selection_locked_before_test"] is True
    assert release["evaluation_protocol_sha256"] == sha256(ROOT / "artifacts/evaluation_protocol.json")
    assert release["domain_plan_sha256"] == sha256(DOMAIN_PLAN_PATH)
    target = DOMAIN_DIRECTORY / "test_evaluation.json"
    if target.exists():
        previous = read_json(target)
        assert previous["release_sha256"] == sha256(ROOT / "artifacts/model_release.json")
        if previous["status"] == "complete":
            print("완료된 동일 모델의 최종 평가를 재사용합니다.")
            return
    prep = read_json(ROOT / "data/aihub_summary/news_briefing_preprocess_report.json")
    path = ROOT / "data/aihub_summary/news_briefing_test.jsonl"
    assert sha256(path) == prep["outputs"]["test"]["sha256"]
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    tokenizer = load_tokenizer()
    pool = domain_pool(records, tokenizer, read_json(DOMAIN_PLAN_PATH)["rules"])
    size = min(protocol["test_size"], len(pool))
    assert size >= 32, "선별 범위의 독립 평가 표본이 부족합니다."
    strata = {}
    for row in pool:
        strata.setdefault(row["doc_type"]+"|"+"+".join(row["categories"]), []).append(row)
    keys = sorted(strata)
    exact = np.array([len(strata[k])*size/len(pool) for k in keys])
    quotas = np.floor(exact).astype(int)
    for i in np.argsort(-(exact-quotas), kind="stable")[:size-int(sum(quotas))]:
        quotas[i] += 1
    rng = np.random.default_rng(protocol["test_seed"])
    selected = []
    for key, quota in zip(keys, quotas):
        values = sorted(strata[key], key=lambda row: row["id"])
        selected += [values[int(i)] for i in rng.permutation(len(values))[:int(quota)]]
    selected.sort(key=lambda r: r["id"])
    assert len(selected) == len({r["group_id"] for r in selected}) == size
    # 기존 전체 train/validation과 교집합을 재검사합니다.
    for field in ["group_id", "source_hash", "doc_ids", "passage_ids", "document_keys"]:
        selected_keys = {v for r in selected for v in (r[field] if isinstance(r[field], list) else [r[field]])}
        for split in ["train", "validation"]:
            with (ROOT / f"data/aihub_summary/news_briefing_{split}.jsonl").open(encoding="utf-8") as stream:
                for line in stream:
                    row = json.loads(line)
                    values = row[field] if isinstance(row[field], list) else [row[field]]
                    assert not selected_keys.intersection(values), (split, field)
    model = load_model(ROOT / release["model_directory"], release["weights_sha256"])
    rows = []
    for i, row in enumerate(selected):
        prediction = generate(row["source"], model, tokenizer, release["generation_settings"], release["max_source_tokens"])
        rows.append({"id": row["id"], "doc_type": row["doc_type"], "categories": row["categories"],
                     "source": row["source"], "target": row["target"], "references": row["reference_summaries"],
                     "lead_sentence": re.split(r"(?<=[.!?])\s+", row["source"], maxsplit=1)[0],
                     "selected_model": prediction["summary"], "generation_seconds": prediction["seconds"],
                     "review_signals": {k: prediction[k] for k in ["numbers_not_in_source", "repeated_word_trigrams"]}})
        if (i+1) % 16 == 0:
            print(f"[최종 평가] 생성 {i+1}/{size}", flush=True)
    metrics = aggregate_metrics(rows, ["lead_sentence", "selected_model"])
    write_json(target, {"status": "complete", "created_at": datetime.now().astimezone().isoformat(),
                   "release_sha256": sha256(ROOT / "artifacts/model_release.json"),
                   "protocol_sha256": sha256(ROOT / "artifacts/evaluation_protocol.json"),
                   "source_sha256": prep["outputs"]["test"]["sha256"], "split": "test",
                   "candidate_groups": len(pool), "n": size, "test_seed": protocol["test_seed"],
                   "metrics": metrics, "doc_types": dict(Counter(r["doc_type"] for r in rows)),
                   "generations": rows, "split_overlap_zero": True,
                   "median_generation_seconds": float(np.median([r["generation_seconds"] for r in rows])),
                   "limits": ["Curated short-document subset, not full test benchmark", "No tuning after this evaluation",
                              "Reference topic terms also participate in frozen scope selection",
                              "ROUGE uses Unicode word tokens without Korean morphological analysis"]})
    print(json.dumps(metrics, ensure_ascii=False, indent=2))


## 6. 중요 문장 선별기와 재작성 생성기

### 문서 요약의 경로와 생성 설정

중요 문장을 먼저 고른 뒤 KoBART가 하나의 요약문을 생성합니다. 시연과 학습에 같은 설정을 사용합니다.

In [ ]:
SELECTOR = ROOT / "artifacts/rewrite_selector.pt"
MANIFEST = ROOT / "artifacts/rewrite_release.json"
REPORT = ROOT / "artifacts/rewrite_validation.json"
DECODER = ROOT / "artifacts/rewrite_decoder.safetensors"
GENERATOR_MANIFEST = ROOT / "artifacts/rewrite_generator.json"
MAX_DOCUMENT_TOKENS = 4096
MAX_EVIDENCE_TOKENS = 480
GENERATION = {"max_new_tokens": 112, "min_new_tokens": 8, "num_beams": 4,
              "do_sample": False, "no_repeat_ngram_size": 3, "length_penalty": 1.1}
REWRITE_SEED = SELECTOR_SETTINGS["seed"]


### 문장 분리와 문맥 벡터 만들기

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def sentence_units(text, tokenizer):
    """문장 끝과 문단을 이용해 나눈다. 너무 긴 한 문장도 버리지 않고 토큰 구간으로 나눈다."""
    units = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = normalize_text(paragraph)
        # 따옴표 자체는 문장 끝이 아니다. ‘제품명’ 다음에서도 문맥을 유지한다.
        parts = re.split(r'(?<=[.!?。])\s+|(?<=[.!?。][”’\"])\s+', paragraph)
        for part in parts:
            if not part.strip():
                continue
            encoded = tokenizer(part, add_special_tokens=False, return_offsets_mapping=True)
            offsets = encoded["offset_mapping"]
            if not offsets:
                continue
            if len(offsets) <= 510:
                units.append(part)
            else:
                start = 0
                for end in range(510, len(offsets) + 510, 510):
                    boundary = offsets[min(end, len(offsets)) - 1][1]
                    units.append(part[start:boundary].strip())
                    start = boundary
    return [u for u in units if u]


def document_features(text, model, tokenizer):
    count = len(input_ids(text, tokenizer))
    if not normalize_text(text):
        raise ValueError("요약할 본문을 입력해 주세요.")
    if count > MAX_DOCUMENT_TOKENS:
        raise ValueError(f"현재 문서는 {count}토큰입니다. 한 주제의 본문을 {MAX_DOCUMENT_TOKENS}토큰 이하로 나눠 주세요. 자동으로 자르지 않습니다.")
    units = sentence_units(text, tokenizer)
    if not units:
        raise ValueError("읽을 수 있는 문장을 찾지 못했습니다.")
    groups, current = [], []
    for i, unit in enumerate(units):
        if current and len(input_ids(" ".join(units[j] for j in [*current, i]), tokenizer)) > 576:
            groups.append(current)
            current = []
        current.append(i)
    if current:
        groups.append(current)
    vectors = [None] * len(units)
    model.eval()
    with torch.inference_mode():
        for group in groups:
            joined = " ".join(units[j] for j in group)
            encoded = tokenizer(joined, add_special_tokens=False, return_offsets_mapping=True)
            ids = torch.tensor([[tokenizer.bos_token_id, *encoded["input_ids"], tokenizer.eos_token_id]],
                               device=next(model.parameters()).device)
            hidden = model.get_encoder()(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state[0]
            cursor = 0
            for j in group:
                end = cursor + len(units[j])
                indices = [k + 1 for k, (a, b) in enumerate(encoded["offset_mapping"]) if b > cursor and a < end]
                if not indices:
                    raise ValueError("문장의 토큰 범위를 찾지 못했습니다.")
                vectors[j] = hidden[indices].mean(0).cpu()
                cursor = end + 1
    embeddings = torch.stack(vectors).clone()
    # 문장 벡터는 같은 구간의 앞뒤 문맥을 포함한다. 문서 평균과 상대 위치도 함께 사용한다.
    document = embeddings.mean(0).expand_as(embeddings)
    metadata = torch.tensor([[i / max(1, len(units) - 1), min(len(u), 500) / 500]
                             for i, u in enumerate(units)], dtype=torch.float32)
    features = torch.cat([embeddings, document, metadata], dim=1)
    return units, features, count, len(groups)


### 중요 문장 선별기 구조와 저장 모델 불러오기

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
class SentenceSelector(nn.Module):
    def __init__(self, hidden=768):
        super().__init__()
        self.layers = nn.Sequential(nn.LayerNorm(hidden * 2 + 2), nn.Linear(hidden * 2 + 2, SELECTOR_SETTINGS["hidden_units"]),
                                    nn.GELU(), nn.Dropout(SELECTOR_SETTINGS["dropout"]), nn.Linear(SELECTOR_SETTINGS["hidden_units"], 1))

    def forward(self, features):
        return self.layers(features).squeeze(-1)


def load_selector():
    if not SELECTOR.exists() or not MANIFEST.exists():
        raise ValueError("핵심 문장 선별 모델을 먼저 준비해 주세요.")
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    if sha256(SELECTOR) != manifest["selector_sha256"]:
        raise ValueError("선별 모델 파일이 준비 기록과 다릅니다.")
    if BASE_RELEASE_FINGERPRINT != manifest["base_release_sha256"]:
        raise ValueError("생성 모델이 선별 모델 학습 당시와 다릅니다.")
    head = SentenceSelector()
    head.load_state_dict(torch.load(SELECTOR, map_location="cpu", weights_only=True))
    return head.eval()


def load_rewriter():
    """기존 가중치를 별도로 불러와 학습한 디코더 두 층만 교체한다."""
    from safetensors.torch import load_file
    release = read_release()
    model = load_model(ROOT / release["model_directory"], release["weights_sha256"])
    if GENERATOR_MANIFEST.exists():
        manifest = json.loads(GENERATOR_MANIFEST.read_text(encoding="utf-8"))
        if manifest["base_release_sha256"] != BASE_RELEASE_FINGERPRINT:
            raise ValueError("추가 학습의 원본 모델이 다릅니다.")
        if manifest["selector_manifest_sha256"] != sha256(MANIFEST) or manifest["weights_sha256"] != sha256(DECODER):
            raise ValueError("추가 학습 모델의 식별값이 다릅니다.")
        state = load_file(str(DECODER))
        expected = {name for name, _ in model.named_parameters() if decoder_parameter(name)}
        if set(state) != expected:
            raise ValueError("추가 학습한 디코더의 구성과 다릅니다.")
        model.load_state_dict(state, strict=False)
    return model.eval()


def decoder_parameter(name):
    return name.startswith(tuple(f"model.decoder.layers.{i}." for i in GENERATOR_SETTINGS["trainable_layers"]))


### 중요 문장 선택과 생성 요약 연결

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def select_evidence(units, scores, tokenizer):
    selected = []
    # 중요도 순으로 예산을 배분한 뒤 원문 순서로 복원한다. 목록을 그대로 결과로 반환하지 않는다.
    for index in sorted(range(len(units)), key=lambda i: (-scores[i], i)):
        proposal = sorted([*selected, index])
        if len(input_ids(" ".join(units[i] for i in proposal), tokenizer)) <= MAX_EVIDENCE_TOKENS:
            selected = proposal
        if len(selected) >= 5:
            break
    if not selected:
        # 한 문장 자체가 예산보다 길면 모델 허용 범위 안에서 그 문장을 온전히 사용한다.
        selected = [max(range(len(units)), key=lambda i: scores[i])]
    return selected, " ".join(units[i] for i in selected)


def rewriting_signals(source, summary):
    compact = lambda value: re.sub(r"\W+", "", value)
    source_key, summary_key = compact(source), compact(summary)
    source_grams = {source_key[i:i+6] for i in range(max(0, len(source_key)-5))}
    grams = [summary_key[i:i+6] for i in range(max(0, len(summary_key)-5))]
    words = summary.split()
    adjacent = sorted({a for a, b in zip(words, words[1:]) if a == b})
    return {"verbatim_source_span": bool(summary_key and summary_key in source_key),
            "adjacent_repeated_words": adjacent,
            "novel_char_6gram_ratio": sum(g not in source_grams for g in grams) / len(grams) if grams else 0.,
            "compression_ratio": len(summary_key) / max(1, len(source_key))}


def summarize_document(text, model, tokenizer, selector):
    start = time.perf_counter()
    units, features, count, chunks = document_features(text, model, tokenizer)
    with torch.inference_mode():
        scores = selector(features).sigmoid().tolist()
    indices, evidence = select_evidence(units, scores, tokenizer)
    result = generate(evidence, model, tokenizer, GENERATION, max_source_tokens=576)
    result.update({"source": normalize_text(text), "source_tokens": count,
                   "evidence_tokens": len(input_ids(evidence, tokenizer)),
                   "selected_sentences": [units[i] for i in indices],
                   "selected_indices": indices, "sentence_count": len(units), "chunks": chunks,
                   "pipeline": "contextual_sentence_selector_then_kobart_generator",
                   "seconds": time.perf_counter() - start})
    result.update(review_signals(text, result["summary"]))
    result.update(rewriting_signals(text, result["summary"]))
    result["numbers_not_in_evidence"] = review_signals(evidence, result["summary"])["numbers_not_in_source"]
    return result


### 선별 학습 자료와 약한 정답 라벨 준비

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def overlap_label(sentence, references):
    def score(reference):
        a, b = Counter(re.findall(r"\w+", sentence)), Counter(re.findall(r"\w+", reference))
        return 2 * sum((a & b).values()) / max(1, sum(a.values()) + sum(b.values()))
    return max(score(r) for r in references)



def selected_rows(split):
    wanted = set(BASE_RESULTS["domain"]["selection"][f"{split}_ids"])
    rows = [row for row in read_training_rows(split) if row["id"] in wanted]
    rows.sort(key=lambda row: row["id"])
    order = np.random.default_rng(REWRITE_SEED).permutation(len(rows))
    return [rows[int(i)] for i in order]


### 중요 문장 선별기 학습

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def train_sentence_selector():
    torch.manual_seed(REWRITE_SEED)
    release = read_release()
    model = load_model(ROOT / release["model_directory"], release["weights_sha256"])
    tokenizer = load_tokenizer()
    train, tuning = selected_rows("train")[:SELECTOR_SETTINGS["train_count"]], selected_rows("validation")[:SELECTOR_SETTINGS["tuning_count"]]
    assert not {r["group_id"] for r in train} & {r["group_id"] for r in tuning}
    features, labels = {}, {}
    for name, rows in (("train", train), ("tuning", tuning)):
        xs, ys = [], []
        for i, row in enumerate(rows):
            units, x, _, _ = document_features(row["source"], model, tokenizer)
            relevance = np.array([overlap_label(u, row["reference_summaries"]) for u in units])
            # 정답 요약은 학습 라벨 생성에만 사용한다. 실행 시 선별 함수에 정답을 전달하지 않는다.
            y = (relevance >= max(SELECTOR_SETTINGS["label_minimum"], relevance.max() * SELECTOR_SETTINGS["label_relative"])).astype(np.float32)
            if not y.any():
                y[relevance.argmax()] = 1
            xs.append(x)
            ys.append(torch.tensor(y))
            if (i + 1) % 32 == 0:
                print(f"[문맥 읽기] {name} {i + 1}/{len(rows)}", flush=True)
        features[name], labels[name] = torch.cat(xs), torch.cat(ys)
    head = SentenceSelector()
    optimizer = torch.optim.AdamW(head.parameters(), lr=SELECTOR_SETTINGS["learning_rate"], weight_decay=SELECTOR_SETTINGS["weight_decay"])
    weight = (1 - labels["train"].mean()) / labels["train"].mean()
    criterion = nn.BCEWithLogitsLoss(pos_weight=weight)
    history, best, best_state = [], float("inf"), None
    for epoch in range(SELECTOR_SETTINGS["epochs"]):
        head.train()
        indices = torch.randperm(len(labels["train"]))
        for batch in indices.split(SELECTOR_SETTINGS["batch_size"]):
            optimizer.zero_grad()
            loss = criterion(head(features["train"][batch]), labels["train"][batch])
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.inference_mode():
            train_loss = criterion(head(features["train"]), labels["train"]).item()
            val_loss = criterion(head(features["tuning"]), labels["tuning"]).item()
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "tuning_loss": val_loss})
        if val_loss < best:
            best = val_loss
            best_state = {k: v.clone() for k, v in head.state_dict().items()}
    torch.save(best_state, SELECTOR)
    write_json(MANIFEST, {"status": "experimental", "architecture": "frozen KoBART encoder + MLP sentence selector + existing fine-tuned KoBART generator",
               "train_count": len(train), "tuning_count": len(tuning), "train_ids": [r["id"] for r in train],
               "tuning_ids": [r["id"] for r in tuning], "selector_sha256": sha256(SELECTOR),
               "base_release_sha256": BASE_RELEASE_FINGERPRINT, "generation_settings": GENERATION,
               "max_document_tokens": MAX_DOCUMENT_TOKENS, "max_evidence_tokens": MAX_EVIDENCE_TOKENS,
               "sentence_policy": "v2_terminal_punctuation_required_before_closing_quote",
               "history": history, "best_epoch": min(history, key=lambda r: r["tuning_loss"])["epoch"],
               "label_limit": "정답 요약과의 어절 유사도로 만든 약한 문장 라벨. 사람이 정한 중요도 정답이 아님.",
               "long_document_limit": "긴 문서는 구간별 문맥으로 읽음. 576토큰을 넘는 문서의 요약 성능은 별도 검증 전.",
               "test_used": False, "settings": SELECTOR_SETTINGS})
    print("선별 모델 준비 완료. 생성 모델의 기존 가중치와 test 기록은 변경하지 않았습니다.", flush=True)


### 생성기 마지막 두 층 추가 학습

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def train_rewriter_generator():
    """정답이 원문의 연속 복사가 아닌 자료로 디코더 마지막 두 층을 추가 학습한다."""
    from safetensors.torch import save_file
    torch.manual_seed(GENERATOR_SETTINGS["seed"])
    release = read_release()
    model = load_model(ROOT / release["model_directory"], release["weights_sha256"]).to(DEVICE)
    tokenizer, selector = load_tokenizer(), load_selector()
    # 학습/검증 분할을 유지한다. 뒤쪽 검증 32건은 학습이나 저장 시점 선택에 사용하지 않는다.
    collections = {"train": selected_rows("train"), "tuning": selected_rows("validation")[:SELECTOR_SETTINGS["tuning_count"]]}
    pairs = {}
    for split, rows in collections.items():
        selected = []
        for row in rows:
            references = [r for r in row["reference_summaries"]
                          if not rewriting_signals(row["source"], r)["verbatim_source_span"]
                          and rewriting_signals(row["source"], r)["novel_char_6gram_ratio"] >= GENERATOR_SETTINGS["minimum_novel_ratio"]]
            if not references:
                continue
            units, features, _, _ = document_features(row["source"], model, tokenizer)
            with torch.inference_mode():
                scores = selector(features).sigmoid().tolist()
            _, evidence = select_evidence(units, scores, tokenizer)
            # 선택된 원문에 없는 숫자를 정답으로 강요하지 않는다.
            references = [r for r in references if not review_signals(evidence, r)["numbers_not_in_source"]]
            if not references:
                continue
            target = references[0]
            labels = tokenizer.encode(normalize_text(target), add_special_tokens=False) + [tokenizer.eos_token_id]
            if len(labels) > GENERATOR_SETTINGS["max_target_tokens"]:
                continue
            selected.append({"id": row["id"], "input": input_ids(evidence, tokenizer), "labels": labels,
                             "evidence": evidence, "target": target})
            if len(selected) % 32 == 0:
                print(f"[재구성 자료] {split} {len(selected)}", flush=True)
            if len(selected) >= (GENERATOR_SETTINGS["train_count"] if split == "train" else GENERATOR_SETTINGS["tuning_count"]):
                break
        pairs[split] = selected
    if not pairs["train"] or not pairs["tuning"]:
        raise RuntimeError("재구성 학습에 적합한 자료가 부족합니다.")
    for name, parameter in model.named_parameters():
        parameter.requires_grad_(decoder_parameter(name))
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=GENERATOR_SETTINGS["learning_rate"], weight_decay=GENERATOR_SETTINGS["weight_decay"])

    def loss_on(row):
        ids = torch.tensor([row["input"]], device=DEVICE)
        labels = torch.tensor([row["labels"]], device=DEVICE)
        return model(input_ids=ids, attention_mask=torch.ones_like(ids), labels=labels, use_cache=False).loss

    def validation_loss():
        model.eval()
        with torch.inference_mode():
            values = [(loss_on(row).item(), len(row["labels"])) for row in pairs["tuning"]]
        return sum(loss * count for loss, count in values) / sum(count for _, count in values)

    initial_loss = validation_loss()
    best_loss = initial_loss
    best_state = {name: parameter.detach().cpu().contiguous().clone()
                  for name, parameter in model.named_parameters() if decoder_parameter(name)}
    history = [{"samples": 0, "validation_loss": initial_loss}]
    start = time.perf_counter()
    for epoch in range(GENERATOR_SETTINGS["epochs"]):
        for offset in range(0, len(pairs["train"]), GENERATOR_SETTINGS["gradient_accumulation"]):
            model.train()
            # 고정된 인코더와 디코더 앞쪽 층의 dropout도 꺼서 문맥 표현을 일정하게 유지한다.
            model.model.encoder.eval()
            for i, layer in enumerate(model.model.decoder.layers):
                if i not in GENERATOR_SETTINGS["trainable_layers"]:
                    layer.eval()
            optimizer.zero_grad()
            batch = pairs["train"][offset:offset+GENERATOR_SETTINGS["gradient_accumulation"]]
            target_count = sum(len(row["labels"]) for row in batch)
            mean_loss = 0.
            for row in batch:
                loss = loss_on(row) * len(row["labels"]) / target_count
                mean_loss += loss.item()
                loss.backward()
            torch.nn.utils.clip_grad_norm_(parameters, GENERATOR_SETTINGS["max_grad_norm"])
            optimizer.step()
            seen = epoch * len(pairs["train"]) + offset + len(batch)
            if seen % GENERATOR_SETTINGS["validation_every_samples"] == 0 or offset + len(batch) == len(pairs["train"]):
                val_loss = validation_loss()
                history.append({"samples": seen, "last_batch_train_loss": mean_loss, "validation_loss": val_loss})
                if val_loss < best_loss:
                    best_loss = val_loss
                    best_state = {name: parameter.detach().cpu().contiguous().clone()
                                  for name, parameter in model.named_parameters() if decoder_parameter(name)}
                print(f"[생성 추가 학습] {seen}/{len(pairs['train'])} 검증 손실 {val_loss:.4f}", flush=True)
    save_file(best_state, str(DECODER))
    write_json(GENERATOR_MANIFEST, {"status": "experimental", "train_count": len(pairs["train"]),
               "tuning_count": len(pairs["tuning"]), "train_ids": [r["id"] for r in pairs["train"]],
               "tuning_ids": [r["id"] for r in pairs["tuning"]], "history": history,
               "base_release_sha256": BASE_RELEASE_FINGERPRINT,
               "selector_manifest_sha256": sha256(MANIFEST), "weights_sha256": sha256(DECODER),
               "generation_settings": GENERATION, "trainable_parameters": sum(p.numel() for p in parameters),
               "trainable_layers": GENERATOR_SETTINGS["trainable_layers"],
               "selected_trained_weights": best_loss < initial_loss,
               "epochs": GENERATOR_SETTINGS["epochs"], "learning_rate": GENERATOR_SETTINGS["learning_rate"],
               "gradient_accumulation": GENERATOR_SETTINGS["gradient_accumulation"], "settings": GENERATOR_SETTINGS,
               "selection": "lowest tuning token-weighted cross entropy; retain initial weights if no improvement",
               "best_validation_loss": best_loss, "initial_validation_loss": initial_loss,
               "training_seconds": time.perf_counter() - start,
               "limit": "정답이 원문의 연속 구간이 아닌 자료를 선별한 소규모 학습. 모든 문서의 재구성/사실성을 보장하지 않음.",
               "test_used": False})
    print("생성 부분 추가 학습 완료", flush=True)


### 재작성 전후 개발 자료 비교

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def evaluate_rewriter():
    report_path = REWRITE_REPORT_PATH
    release = read_release()
    tokenizer = load_tokenizer()
    model = load_model(ROOT / release["model_directory"], release["weights_sha256"])
    rewriter = load_rewriter() if GENERATOR_MANIFEST.exists() else model
    selector = load_selector()
    rows = selected_rows("validation")[FACTUAL_SETTINGS["tuning_count"]:FACTUAL_SETTINGS["tuning_count"] + FACTUAL_SETTINGS["development_count"]]
    results = []
    for i, row in enumerate(rows):
        baseline = generate(row["source"], model, tokenizer, release["generation_settings"])
        candidate = summarize_document(row["source"], rewriter, tokenizer, selector)
        results.append({"id": row["id"], "source": row["source"], "references": row["reference_summaries"],
                        "before": {**baseline, **rewriting_signals(row["source"], baseline["summary"]),
                                   "metrics": rouge_scores(baseline["summary"], row["reference_summaries"])},
                        "after": {**candidate, "metrics": rouge_scores(candidate["summary"], row["reference_summaries"])}})
        print(f"[생성 비교] {i+1}/{len(rows)}", flush=True)
    aggregate = {}
    for mode in ("before", "after"):
        aggregate[mode] = {metric: float(np.mean([r[mode]["metrics"][metric] for r in results]))
                           for metric in ("rouge1", "rouge2", "rougeL", "char_lcs_f1")}
        aggregate[mode].update({"verbatim_count": sum(r[mode]["verbatim_source_span"] for r in results),
                               "novel_number_count": sum(bool(r[mode]["numbers_not_in_source"]) for r in results),
                               "repetition_count": sum(bool(r[mode]["repeated_word_trigrams"]) for r in results),
                               "adjacent_repetition_count": sum(bool(rewriting_signals(r[mode]["source"], r[mode]["summary"])["adjacent_repeated_words"]) for r in results),
                               "median_seconds": float(np.median([r[mode]["seconds"] for r in results]))})
    write_json(report_path, {"scope": "development validation; not a new held-out final test", "count": len(rows),
               "selector_manifest_sha256": sha256(MANIFEST), "aggregate": aggregate, "examples": results,
               "generator_manifest_sha256": sha256(GENERATOR_MANIFEST) if GENERATOR_MANIFEST.exists() else None,
               "interpretation": "문자/어절 겹침은 사실 정확도나 자연스러움 점수가 아님. 새 표현 비율도 단독으로 품질을 뜻하지 않음."})
    print(json.dumps(aggregate, ensure_ascii=False, indent=2), flush=True)


## 7. 주체·반복 확인과 학습률·epoch 비교

### 후보 생성과 관계 대조 설정

모든 실험에서 같은 생성 길이와 후보 수를 사용합니다. 관계 점수는 후보 선택용 보조 신호입니다.

In [ ]:
FACTUAL_DIRECTORY = ROOT / 'artifacts/factual'
FACTUAL_RELEASE = FACTUAL_DIRECTORY / 'service_release.json'
NLI_DIRECTORY = ROOT / 'artifacts/klue_nli'
CANDIDATE_SETTINGS = {**GENERATION, 'num_return_sequences': 4,
            'return_dict_in_generate': True, 'output_scores': True}


### 원문의 주체와 반복 표현 확인

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def subject_anchor(text):
    text = re.sub(r'^(?:또한|아울러|한편|이에|특히)[, ]+', '', text.strip())
    match = re.match(r'^[\s“‘\"\'(]*([가-힣A-Za-z0-9&·]{2,30}?)(?:에서는|에서|은|는|이|가|의)(?=[\s‘“\"\'])', text)
    return match.group(1) if match else None


def anchored_premise(source, summary, strict_names=True):
    """명시적인 첫 주체가 있는 경우 해당 주체의 원문과 바로 이어진 지시문만 대조한다."""
    anchor = subject_anchor(summary)
    if not anchor:
        return source, None
    sentences = re.split(r'(?<=[.!?。])\s+', normalize_text(source))
    # '카카오'와 '카카오페이'처럼 이름 일부만 같은 다른 주체를 연결하지 않는다.
    name_pattern = re.compile(r'(?<![\w])'+re.escape(anchor)+r'(?=$|[\s.,“‘”’\"\'()]|(?:은|는|이|가|의|와|과|에서|에게|을|를|도|로)(?=\s|[.,]))')
    chosen = [i for i,s in enumerate(sentences) if (name_pattern.search(s) if strict_names else anchor in s)]
    if not chosen:
        return source, None
    for i in list(chosen):
        if i+1 < len(sentences) and re.match(r'^(?:이 회사|해당 회사|이 제품|해당 제품|이 서비스|이를 통해|이에 따라)', sentences[i+1]):
            chosen.append(i+1)
    return ' '.join(sentences[i] for i in sorted(set(chosen))), anchor


def adjacent_words(text):
    return [re.sub(r'^[\W_]+|[\W_]+$', '',w) for w in text.split()]


def punctuation_repetition(text):
    words=adjacent_words(text)
    return sorted({a for a,b in zip(words,words[1:]) if a and a==b})


def multiple_subject_risk(evidence, summary):
    anchor=subject_anchor(summary)
    if not anchor:
        return False
    generic={'이들','그들','우리','기존','홈페','정부','기업','회사','제품','서비스','소비자','고객',
             '전문가들','전문','올해','내년','지난해','국민','당국','업체','이것','그것'}
    if anchor in generic:
        return False
    subjects={subject_anchor(s) for s in re.split(r'(?<=[.!?。])\s+',normalize_text(evidence))}
    subjects.discard(None)
    return bool(subjects-generic-{anchor})


def choose_candidate(evidence,candidates):
    """정상 1순위는 보존한다. NLI 점수 상승 자체를 재작성 이유로 삼지 않는다."""
    for candidate in candidates:
        repeated=bool(punctuation_repetition(candidate['summary']) or candidate['repeated_word_trigrams'])
        candidate['punctuation_insensitive_repeated_words']=punctuation_repetition(candidate['summary'])
        candidate['ranking_score']=(candidate['support']-candidate['contradiction']-float(repeated)
            -float(bool(candidate['numbers_not_in_evidence']))+.05*candidate['generation_score'])
    first=candidates[0]
    risk=multiple_subject_risk(evidence,first['summary'])
    if (not first['punctuation_insensitive_repeated_words'] and not first['repeated_word_trigrams']
        and not first['numbers_not_in_evidence'] and (not risk or first['support']>=.5)):
        return 0
    return max(range(len(candidates)),key=lambda i:candidates[i]['ranking_score'])


### 한국어 문장 관계 모델로 근거 점수 계산

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
class SupportScorer:
    """원문 구간이 생성문을 뒷받침하는지 예측. 점수는 사실 정확도가 아니다."""
    def __init__(self, device='cpu', scope_mode='subject_local_v2'):
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.tokenizer = AutoTokenizer.from_pretrained(NLI_DIRECTORY, local_files_only=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            NLI_DIRECTORY, local_files_only=True, trust_remote_code=False).to(device).eval()
        labels = {str(v).lower(): int(k) for k, v in self.model.config.id2label.items()}
        if not {'entailment', 'contradiction'} <= labels.keys():
            raise RuntimeError(f'NLI 라벨을 확인할 수 없습니다: {labels}')
        self.entailment, self.contradiction = labels['entailment'], labels['contradiction']
        self.device = device
        if scope_mode not in ['global_and_subject_v1','subject_local_v2']:
            raise ValueError('알 수 없는 관계 대조 버전입니다.')
        self.scope_mode = scope_mode

    def _score_windows(self, source, summaries):
        # 처음 512토큰만 잘라 버리지 않는다. 겹치는 모든 구간을 검사한다.
        ids = self.tokenizer.encode(normalize_text(source), add_special_tokens=False)
        windows = [self.tokenizer.decode(ids[i:i+320], skip_special_tokens=True)
                   for i in range(0, len(ids), 224)]
        if not windows:
            windows = ['']
        pairs = [(w, s) for s in summaries for w in windows]
        values = []
        with torch.inference_mode():
            for offset in range(0, len(pairs), 4):
                batch = pairs[offset:offset+4]
                encoded = self.tokenizer([p[0] for p in batch], [p[1] for p in batch],
                    padding=True, truncation='only_first', max_length=510, return_tensors='pt').to(self.device)
                # 이 공개 체크포인트는 BertTokenizer + type_vocab_size=1 RoBERTa다.
                # 두 번째 문장의 segment=1을 전달하면 임베딩 범위를 벗어난다.
                encoded.pop('token_type_ids', None)
                values.extend(self.model(**encoded).logits.softmax(-1).cpu().tolist())
        scores = []
        for i in range(len(summaries)):
            chunk = values[i*len(windows):(i+1)*len(windows)]
            best = max(chunk, key=lambda p: p[self.entailment]-p[self.contradiction])
            scores.append({'support': best[self.entailment], 'contradiction': best[self.contradiction]})
        return scores

    def score(self, source, summaries):
        global_scores = self._score_windows(source, summaries)
        for summary, score in zip(summaries, global_scores):
            # 같은 원문에 나온다는 것만으로 기업과 속성의 연결을 인정하지 않는다.
            # 명시적 주체가 없는 문장/복잡한 공지시 등은 이 규칙으로 보장할 수 없다.
            claims = re.split(r'(?<=[.!?。])\s+', summary)
            anchors, scoped = [], []
            for claim in claims:
                premise, anchor = anchored_premise(source, claim,strict_names=self.scope_mode=='subject_local_v2')
                if anchor:
                    anchors.append(anchor)
                    scoped.append(self._score_windows(premise, [claim])[0])
            score['global_support'] = score['support']
            score['subject_anchors'] = anchors
            if scoped:
                # 긴 원문 전체는 명백히 맞는 인용도 중립으로 오판할 수 있다.
                # 명시적 주체가 있는 주장은 해당 주체의 근거로 판단한다.
                score['support'] = min([s['support'] for s in scoped]+([score['support']] if self.scope_mode=='global_and_subject_v1' else []))
                score['contradiction'] = max([s['contradiction'] for s in scoped]+([score['contradiction']] if self.scope_mode=='global_and_subject_v1' else []))
        return global_scores


### 같은 근거로 요약 후보 생성과 선택

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def generate_candidates(evidence, model, tokenizer):
    ids = input_ids(evidence, tokenizer)
    if len(ids) > 576:
        raise ValueError('선별 근거가 입력 범위를 넘었습니다.')
    tensor = torch.tensor([ids], device=next(model.parameters()).device)
    with torch.inference_mode():
        output = model.generate(input_ids=tensor, attention_mask=torch.ones_like(tensor),
                                use_cache=True, **CANDIDATE_SETTINGS)
    return [{'summary': tokenizer.decode(seq, skip_special_tokens=True,
                                        clean_up_tokenization_spaces=False).strip(),
             'generation_score': float(score)}
            for seq, score in zip(output.sequences, output.sequences_scores)]


def rank_candidates(source, evidence, candidates, scorer, conservative=None):
    support = scorer.score(source, [c['summary'] for c in candidates])
    enriched = []
    for candidate, nli in zip(candidates, support):
        text = candidate['summary']
        signals = {**review_signals(source, text), **rewriting_signals(source, text)}
        unsupported_numbers = review_signals(evidence, text)['numbers_not_in_source']
        repetition = bool(signals['adjacent_repeated_words'] or signals['repeated_word_trigrams'])
        # 이름별 예외/결과 문자열 수정은 없다. 모든 문서에 동일한 후보 선택 규칙.
        rank = (nli['support'] - nli['contradiction'] - 1.0*repetition
                - 1.0*bool(unsupported_numbers) + .05*candidate['generation_score'])
        enriched.append({**candidate, **signals, **nli, 'ranking_score': rank,
                         'numbers_not_in_evidence': unsupported_numbers})
    if conservative is None:
        conservative=getattr(scorer,'scope_mode','subject_local_v2')!='global_and_subject_v1'
    best = choose_candidate(evidence,enriched) if conservative else max(range(len(enriched)),key=lambda i:enriched[i]['ranking_score'])
    return enriched, best


### 근거가 약한 후보의 조건부 재생성

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def repair_candidates(source, evidence, assessed, best, model, tokenizer, scorer, conservative=True):
    """모든 후보의 근거가 약하면 해당 주체의 근거만으로 한 번 더 생성한다.

    같은 규칙/최대 1회/4개 후보를 전후 모두 적용한다. 잘못된 문구를 문자열 치환하지 않는다.
    """
    first=assessed[0]
    if conservative and (not multiple_subject_risk(evidence,first['summary'])
        and not punctuation_repetition(first['summary']) and not first['repeated_word_trigrams']
        and not first['numbers_not_in_evidence']):
        return assessed,best,None
    if any(c['support'] >= .5 and not (punctuation_repetition(c['summary']) if conservative else c['adjacent_repeated_words'])
           and not c['repeated_word_trigrams']
           and not c['numbers_not_in_evidence'] for c in assessed):
        return assessed, best, None
    # 선별 과정에서 주체의 소개 문장이 빠졌다면 원문에서 함께 복원한다.
    # 기존 비교 정책은 선택된 근거만 사용하며, 새 정책은 양쪽 모델 모두 원문을 사용한다.
    premise, anchor = anchored_premise(source if conservative else evidence, assessed[best]['summary'])
    if len(input_ids(premise,tokenizer)) > 576:
        premise,anchor=anchored_premise(evidence,assessed[best]['summary'])
    if not anchor or normalize_text(premise) == normalize_text(evidence):
        return assessed, best, None
    if not premise.strip() or len(input_ids(premise,tokenizer)) > 576:
        return assessed, best, None
    proposed = generate_candidates(premise,model,tokenizer)
    repaired, _ = rank_candidates(source,premise,proposed,scorer,conservative=conservative)
    for c in repaired:
        c['repair_evidence'] = premise
    merged = [*assessed,*repaired]
    choice = choose_candidate(evidence,merged) if conservative else max(range(len(merged)),key=lambda i:merged[i]['ranking_score'])
    return merged, choice, {'anchor':anchor,'premise':premise,'generated_candidates':len(repaired)}


### 연속 중복 단어 정리와 결과 기록

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def clean_accidental_repetition(source, summary):
    """원문에 없는 인접 동일 단어만 정리한다. 원래 생성문과 편집 내역은 보존한다."""
    source_words=adjacent_words(normalize_text(source))
    source_pairs=set(zip(source_words,source_words[1:]))
    words, edits = [], []
    for word in summary.split():
        key=adjacent_words(word)[0]
        previous=adjacent_words(words[-1])[0] if words else None
        if (words and key == previous and len(key)>1 and not key.isnumeric()
                and (key,key) not in source_pairs):
            edits.append(key)
            words[-1]=word
        else:
            words.append(word)
    return ' '.join(words), edits


def finish_candidate(source, candidate, scorer):
    result = dict(candidate)
    cleaned, edits = clean_accidental_repetition(source,result['summary'])
    result['raw_generated_summary'] = result['summary']
    result['repetition_cleanup'] = edits
    if edits:
        result['summary'] = cleaned
        result.update(review_signals(source,cleaned))
        result.update(rewriting_signals(source,cleaned))
        result.update(scorer.score(source,[cleaned])[0])
    result['punctuation_insensitive_repeated_words']=punctuation_repetition(result['summary'])
    return result


### 서비스에서 사용하는 모델과 요약 함수

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def load_factual_model():
    from safetensors.torch import load_file

    release = json.loads(FACTUAL_RELEASE.read_text(encoding='utf-8'))
    if sha256(GENERATOR_MANIFEST) != release['initial_generator_manifest_sha256']:
        raise ValueError('비교 시작 모델이 학습 기록과 다릅니다.')
    path = FACTUAL_DIRECTORY / release['checkpoint']
    if sha256(path) != release['weights_sha256']:
        raise ValueError('개선 모델의 식별값이 다릅니다.')
    model = load_rewriter()
    if release.get('use_initial_model', False):
        return model.eval()
    state = load_file(str(path))
    if set(state) != {n for n, _ in model.named_parameters() if decoder_parameter(n)}:
        raise ValueError('개선 모델의 층 구성이 다릅니다.')
    model.load_state_dict(state, strict=False)
    return model.eval()


def summarize_factual(text, model, tokenizer, selector, scorer):
    start = time.perf_counter()
    units, features, count, chunks = document_features(text, model, tokenizer)
    with torch.inference_mode():
        scores = selector(features).sigmoid().tolist()
    indices, evidence = select_evidence(units, scores, tokenizer)
    candidates = generate_candidates(evidence, model, tokenizer)
    assessed, best = rank_candidates(text, evidence, candidates, scorer)
    assessed, best, repair = repair_candidates(text,evidence,assessed,best,model,tokenizer,scorer)
    result = finish_candidate(text,assessed[best],scorer)
    used_evidence = result.get('repair_evidence',evidence)
    result.update({'source': normalize_text(text), 'source_tokens': count,
                   'selected_sentences': re.split(r'(?<=[.!?。])\s+',used_evidence),
                   'initial_selected_sentences': [units[i] for i in indices],
                   'selected_indices': [i for i in range(len(units)) if units[i] in used_evidence],
                   'initial_selected_indices': indices, 'sentence_count': len(units), 'chunks': chunks,
                   'evidence_tokens': len(input_ids(used_evidence, tokenizer)),
                   'seconds': time.perf_counter()-start, 'candidate_index': best,
                   'pipeline': 'trained_rewriter_with_support_reranking',
                   'repair_used': 'repair_evidence' in result, 'repair_attempted':repair is not None,
                   'requires_review': result['support'] < .5 or result['contradiction'] > .25,
                   'support_limit': '한국어 문장 관계 모델의 보조 신호이며 사실 보증이 아닙니다.'})
    return result


### 추가 학습 설정

학습률 1e-5 / 2e-5 / 5e-5를 같은 시작 가중치에서 각각 3 epoch 비교합니다.

In [ ]:
FACTUAL_SEED = FACTUAL_SETTINGS["seed"]
LEARNING_RATES = FACTUAL_SETTINGS["learning_rates"]
EPOCHS = FACTUAL_SETTINGS["epochs"]


def factual_parameter(name):
    return name.startswith(tuple(f"model.decoder.layers.{i}." for i in FACTUAL_SETTINGS["trainable_layers"]))


### MLflow 기록과 난수 시드 설정

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def setup_tracking():
    FACTUAL_DIRECTORY.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri('sqlite:///' + (FACTUAL_DIRECTORY / 'mlflow.db').as_posix())
    name = 'korean-summary-factual-small'
    experiment = mlflow.get_experiment_by_name(name)
    if experiment is None:
        experiment_id = mlflow.create_experiment(name, artifact_location=(FACTUAL_DIRECTORY / 'mlartifacts').as_uri())
    else:
        experiment_id = experiment.experiment_id
    mlflow.set_experiment(experiment_id=experiment_id)


def set_factual_seed():
    random.seed(FACTUAL_SEED)
    np.random.seed(FACTUAL_SEED)
    torch.manual_seed(FACTUAL_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(FACTUAL_SEED)


### 추가 학습·검증·개발 자료 고정

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def prepare_factual_data(device):
    model = load_rewriter().to(device)
    tokenizer, selector = load_tokenizer(), load_selector()
    generator = read_json(GENERATOR_MANIFEST)
    train_lookup = {r['id']: r for r in selected_rows('train')}
    val_lookup = {r['id']: r for r in selected_rows('validation')}
    groups = {
        'train': [train_lookup[i] for i in generator['train_ids']],
        'tuning': selected_rows('validation')[:FACTUAL_SETTINGS['tuning_count']],
        'development': selected_rows('validation')[FACTUAL_SETTINGS['tuning_count']:FACTUAL_SETTINGS['tuning_count'] + FACTUAL_SETTINGS['development_count']],
    }
    result = {}
    for split, rows in groups.items():
        result[split] = []
        for i, row in enumerate(rows):
            units, features, _, _ = document_features(row['source'], model, tokenizer)
            with torch.inference_mode():
                scores = selector(features).sigmoid().tolist()
            indices, evidence = select_evidence(units, scores, tokenizer)
            refs = row['reference_summaries']
            # 기존 추가 학습과 동일한 순서로 재구성 정답을 선택한다.


            valid = [r for r in refs if not rewriting_signals(row['source'], r)['verbatim_source_span']
                     and rewriting_signals(row['source'], r)['novel_char_6gram_ratio'] >= GENERATOR_SETTINGS["minimum_novel_ratio"]
                     and not review_signals(evidence, r)['numbers_not_in_source']]
            target = valid[0] if valid else refs[0]
            result[split].append({'id': row['id'], 'group_id': row['group_id'], 'source': row['source'],
                'references': refs, 'target': target, 'evidence': evidence,
                'selected_indices': indices, 'input': input_ids(evidence, tokenizer),
                'labels': tokenizer.encode(normalize_text(target), add_special_tokens=False)+[tokenizer.eos_token_id]})
            if (i+1) % 32 == 0:
                print(f'[자료 고정] {split} {i+1}/{len(rows)}', flush=True)
    for a, b in [('train','tuning'),('train','development'),('tuning','development')]:
        assert not {r['group_id'] for r in result[a]} & {r['group_id'] for r in result[b]}
    write_json(FACTUAL_DATA_FILE, result)
    del model
    return result


### 학습할 디코더와 고정 인코더 캐시 준비

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def decoder_state(model):
    return {n: p.detach().cpu().contiguous().clone() for n,p in model.named_parameters() if factual_parameter(n)}


def build_cache(model, collections, device):
    result = {}
    model.eval()
    for split in ['train','tuning']:
        cache = []
        for row in collections[split]:
            ids = torch.tensor([row['input']], device=device)
            with torch.no_grad():
                hidden = model.get_encoder()(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state.detach()
            cache.append((hidden, torch.tensor([row['labels']], device=device)))
        result[split] = cache
    return result


### 정답 손실과 반복 벌점 계산

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def output_loss(model, cached, training):
    hidden, labels = cached
    output = model(encoder_outputs=BaseModelOutput(last_hidden_state=hidden),
                   attention_mask=torch.ones(hidden.shape[:2], dtype=torch.long, device=hidden.device),
                   labels=labels, use_cache=False)
    penalty = output.loss.new_zeros(())
    if training:
        # 이미 끝낸 2-token 구절을 같은 문맥에서 다시 이어가는 확률을 낮춘다.
        # 정답에 실제로 반복되는 구절/특수 토큰은 벌점을 주지 않는다.
        ids = labels[0].tolist()
        positions, bad = [], []
        for t in range(2,len(ids)):
            for j in range(t-1):
                if ids[j] == ids[t-1] and ids[j+1] != ids[t] and ids[j+1] > 4:
                    positions.append(t)
                    bad.append(ids[j+1])
        if positions:
            probs = output.logits[0,positions].softmax(-1)
            values = probs.gather(1,torch.tensor(bad,device=labels.device)[:,None]).squeeze(1)
            penalty = -torch.log1p(-values.clamp(max=1-1e-6)).mean()
    return output.loss + FACTUAL_SETTINGS["repeat_penalty_weight"]*penalty, output.loss, penalty


def validation_loss(model, cache):
    model.eval()
    total, count = 0.,0
    with torch.inference_mode():
        for row in cache:
            _, ce, _ = output_loss(model,row,False)
            n = row[1].numel()
            total += ce.item()*n
            count += n
    return total/count


### 후보별 생성 평가와 결과 집계

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def evaluate_rows(model, tokenizer, scorer, rows):
    results=[]
    model.eval()
    for row in rows:
        start=time.perf_counter()
        candidates=generate_candidates(row['evidence'],model,tokenizer)
        generation_seconds=time.perf_counter()-start
        candidates,best=rank_candidates(row['source'],row['evidence'],candidates,scorer)
        total_seconds=time.perf_counter()-start
        for c in candidates:
            c['metrics']=rouge_scores(c['summary'],row['references'])
        results.append({'id':row['id'],'source':row['source'],'references':row['references'],
                        'evidence':row['evidence'],'candidates':candidates,'top1':0,'reranked':best,
                        'generation_seconds':generation_seconds,'total_seconds':total_seconds})
    return results


def aggregate_candidates(rows,mode):
    chosen=[r['candidates'][r[mode]] for r in rows]
    result={k:float(np.mean([r['metrics'][k] for r in chosen])) for k in ['rouge1','rouge2','rougeL','char_lcs_f1']}
    result.update({'count':len(chosen),
        'adjacent_repetition_count':sum(bool(r['adjacent_repeated_words']) for r in chosen),
        'repeated_trigram_count':sum(bool(r['repeated_word_trigrams']) for r in chosen),
        'new_number_count':sum(bool(r['numbers_not_in_source']) for r in chosen),
        'verbatim_count':sum(r['verbatim_source_span'] for r in chosen),
        'mean_support_signal':float(np.mean([r['support'] for r in chosen])),
        'low_support_count':sum(r['support']<.5 for r in chosen),
        'mean_output_characters':float(np.mean([len(r['summary']) for r in chosen])),
        'median_seconds':float(np.median([r['total_seconds' if mode=='reranked' else 'generation_seconds'] for r in rows]))})
    return result


### 학습률과 epoch 비교 학습

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def train_factual_grid(device):
    setup_tracking()
    data=read_json(FACTUAL_DATA_FILE)
    protocol = {'data_sha256': file_fingerprint(FACTUAL_DATA_FILE),
                'initial_generator_manifest_sha256': file_fingerprint(GENERATOR_MANIFEST),
                'settings': FACTUAL_SETTINGS}
    write_json(FACTUAL_DIRECTORY / 'protocol.json', protocol)
    assert sha256(FACTUAL_DATA_FILE) == protocol['data_sha256']
    assert sha256(GENERATOR_MANIFEST) == protocol['initial_generator_manifest_sha256']
    model,tokenizer=load_rewriter().to(device),load_tokenizer()
    scorer=SupportScorer(device,scope_mode='global_and_subject_v1')
    for name,p in model.named_parameters():
        p.requires_grad_(factual_parameter(name))
    initial=decoder_state(model)
    cache=build_cache(model,data,device)
    initial_loss=validation_loss(model,cache['tuning'])
    results=[]
    for lr in LEARNING_RATES:
        result_file=FACTUAL_DIRECTORY/f'lr_{lr:g}.json'
        set_factual_seed()
        model.load_state_dict(initial,strict=False)
        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr,weight_decay=FACTUAL_SETTINGS["weight_decay"])
        epochs=[]
        with mlflow.start_run(run_name=f'lr={lr:g}_epochs=1-{EPOCHS}_{device}') as run:
            mlflow.log_params({'learning_rate':lr,'max_epochs':EPOCHS,'train_count':len(data['train']),
                'tuning_count':len(data['tuning']),'generation_tuning_count':FACTUAL_SETTINGS['generation_tuning_count'],'seed':FACTUAL_SEED,'device':device,
                'gradient_accumulation':FACTUAL_SETTINGS['gradient_accumulation'],'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
                'repeat_unlikelihood_weight':FACTUAL_SETTINGS['repeat_penalty_weight'],'num_beams':4,'max_new_tokens':112,'data_sha256':protocol['data_sha256'],
                'initial_weights':sha256(DECODER),
                'gpu_name':torch.cuda.get_device_name(0) if device=='cuda' else 'none',
                'torch_version':torch.__version__})
            mlflow.log_metric('validation_loss',initial_loss,step=0)
            mlflow.log_artifact(str(FACTUAL_DIRECTORY/'protocol.json'))
            for epoch in range(1,EPOCHS+1):
                start=time.perf_counter()
                order=np.random.default_rng(FACTUAL_SEED+epoch).permutation(len(cache['train']))
                model.train()
                model.model.encoder.eval()
                for i, layer in enumerate(model.model.decoder.layers):
                    if i not in FACTUAL_SETTINGS['trainable_layers']:
                        layer.eval()
                ce_total,ul_total,tokens=0.,0.,0
                for offset in range(0,len(order),FACTUAL_SETTINGS["gradient_accumulation"]):
                    batch=[cache['train'][int(i)] for i in order[offset:offset+FACTUAL_SETTINGS["gradient_accumulation"]]]
                    target_tokens=sum(r[1].numel() for r in batch)
                    optimizer.zero_grad(set_to_none=True)
                    for row in batch:
                        loss,ce,ul=output_loss(model,row,True)
                        n=row[1].numel()
                        (loss*n/target_tokens).backward()
                        ce_total+=ce.item()*n
                        ul_total+=ul.item()*n
                        tokens+=n
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], FACTUAL_SETTINGS["max_grad_norm"])
                    optimizer.step()
                val=validation_loss(model,cache['tuning'])
                checkpoint=f'lr_{lr:g}_epoch_{epoch}.safetensors'
                save_file(decoder_state(model),str(FACTUAL_DIRECTORY/checkpoint))
                generated=evaluate_rows(model,tokenizer,scorer,data['tuning'][:FACTUAL_SETTINGS['generation_tuning_count']])
                metrics={'epoch':epoch,'learning_rate':lr,'train_loss':ce_total/tokens,
                    'repeat_unlikelihood':ul_total/tokens,'validation_loss':val,
                    'seconds':time.perf_counter()-start,'checkpoint':checkpoint,
                    'weights_sha256':sha256(FACTUAL_DIRECTORY/checkpoint),'mlflow_run_id':run.info.run_id,
                    'top1':aggregate_candidates(generated,'top1'),'reranked':aggregate_candidates(generated,'reranked')}
                epochs.append(metrics)
                results.append(metrics)
                mlflow.log_metrics({k:metrics[k] for k in ['train_loss','validation_loss','repeat_unlikelihood','seconds']},step=epoch)
                for mode in ['top1','reranked']:
                    mlflow.log_metrics({f'{mode}_{k}':v for k,v in metrics[mode].items()},step=epoch)
                write_json(FACTUAL_DIRECTORY/f'lr_{lr:g}_epoch_{epoch}_tuning.json',generated)
                write_json(result_file,{'epochs':epochs,'device':device,'initial_validation_loss':initial_loss})
                print(f'[학습 비교] lr={lr:g} epoch={epoch} val={val:.4f} repeat={metrics["top1"]["adjacent_repetition_count"]} seconds={metrics["seconds"]:.1f}',flush=True)
            mlflow.log_artifact(str(result_file))
    best=min(results,key=lambda r:(r['validation_loss'],r['epoch'],r['learning_rate']))
    release={'status':'experimental_checkpoint_selected_on_tuning_only',**best,
        'initial_validation_loss':initial_loss,'initial_generator_manifest_sha256':sha256(GENERATOR_MANIFEST),
        'selector_manifest_sha256':sha256(MANIFEST),'generation_settings':CANDIDATE_SETTINGS,
        'adopted_in_app':False}
    if best['validation_loss']>=initial_loss:
        release['status']='no_validation_loss_improvement'
    write_json(FACTUAL_DIRECTORY/'release.json',release)
    write_json(FACTUAL_DIRECTORY/'grid.json',{'initial_validation_loss':initial_loss,'epochs':results,'selected':best})
    print('[학습률/회차 비교 완료]',json.dumps(best,ensure_ascii=False),flush=True)


### 추가 학습 전후 동일 조건 비교

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def compare_factual_models(device):
    setup_tracking()
    path=COMPARISON_DIRECTORY/'comparison.json'
    data=read_json(FACTUAL_DATA_FILE)
    release=read_json(FACTUAL_MODEL_DIRECTORY/'release.json')
    tokenizer,scorer=load_tokenizer(),SupportScorer(device,scope_mode='global_and_subject_v1')
    model=load_rewriter().to(device)
    rows=data['development']
    # 동일 장비에서 준비 실행 후 측정하며 모든 방식의 근거/생성 길이/beam을 고정한다.
    generate_candidates(data['tuning'][0]['evidence'],model,tokenizer)
    before=evaluate_rows(model,tokenizer,scorer,rows)
    print('[동일 조건 비교] before 완료',flush=True)
    assert sha256(FACTUAL_MODEL_DIRECTORY/release['checkpoint'])==release['weights_sha256']
    model.load_state_dict(load_file(str(FACTUAL_MODEL_DIRECTORY/release['checkpoint'])),strict=False)
    generate_candidates(data['tuning'][0]['evidence'],model,tokenizer)
    after=evaluate_rows(model,tokenizer,scorer,rows)
    report={'scope':'fixed 32 development documents; not a fresh held-out test',
        'count':len(rows),'device':device,'generation_settings':CANDIDATE_SETTINGS,
        'data_sha256':sha256(FACTUAL_DATA_FILE),'release_sha256':sha256(FACTUAL_MODEL_DIRECTORY/'release.json'),
        'aggregate':{f'{stage}_{mode}':aggregate_candidates(values,mode) for stage,values in [('before',before),('after',after)] for mode in ['top1','reranked']},
        'examples':[{'id':b['id'],'before':b,'after':a} for b,a in zip(before,after)],
        'limit':'NLI reranking and NLI scores share a model; a higher support signal is not independent evidence of factual accuracy. Manual source comparison required.'}
    write_json(path,report)
    with mlflow.start_run(run_name=f'fixed_comparison_32_{device}'):
        mlflow.log_params({'count':len(rows),'device':device,'selected_lr':release['learning_rate'],
                          'selected_epoch':release['epoch'],'same_evidence':True,'same_generation':True})
        for arm,metrics in report['aggregate'].items():
            mlflow.log_metrics({f'{arm}_{k}':v for k,v in metrics.items()})
        mlflow.log_artifact(str(path))
    print(json.dumps(report['aggregate'],ensure_ascii=False,indent=2),flush=True)


### 같은 재생성 예산으로 전후 비교

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def compare_repairs():
    path=COMPARISON_DIRECTORY/'comparison_repair.json'
    original=json.loads((COMPARISON_DIRECTORY/'comparison.json').read_text(encoding='utf-8'))
    release=json.loads((FACTUAL_MODEL_DIRECTORY/'release.json').read_text(encoding='utf-8'))
    device=original['device']
    model,tokenizer,scorer=load_rewriter().to(device),load_tokenizer(),SupportScorer(device)
    report={'scope':original['scope'],'count':original['count'],'device':device,
        'base_comparison_sha256':sha256(COMPARISON_DIRECTORY/'comparison.json'),'support_scorer_version':'subject_local_v2',
        'repair_protocol':{'same_rule_before_after':True,'trigger':'no supported non-repetitive candidate',
           'max_extra_passes':1,'extra_candidates':4,'max_new_tokens':112,'num_beams':4,
           'input':'same selected evidence restricted to generated explicit subject; no references used',
           'limit':'may omit other subjects; requires source review; NLI is not factual ground truth'},
        'examples':[],'aggregate':{}}
    arms={}
    for stage in ['before','after']:
        if stage=='after':
            assert sha256(FACTUAL_MODEL_DIRECTORY/release['checkpoint'])==release['weights_sha256']
            model.load_state_dict(load_file(str(FACTUAL_MODEL_DIRECTORY/release['checkpoint'])),strict=False)
        values=[]
        unrepaired=[]
        for i,pair in enumerate(original['examples']):
            row=copy.deepcopy(pair[stage])
            start=time.perf_counter()
            # 고정된 동일 생성문들을 양쪽 모두 같은 최신 대조 규칙으로 다시 평가한다.
            clean=[{'summary':c['summary'],'generation_score':c['generation_score']} for c in row['candidates']]
            assessed,selected=rank_candidates(row['source'],row['evidence'],clean,scorer,conservative=False)
            for c in assessed:
                c['metrics']=rouge_scores(c['summary'],row['references'])
            row.update({'candidates':assessed,'reranked':selected})
            row['total_seconds']=row['generation_seconds']+time.perf_counter()-start
            unrepaired.append(copy.deepcopy(row))
            start=time.perf_counter()
            candidates,best,repair=repair_candidates(row['source'],row['evidence'],row['candidates'],row['reranked'],model,tokenizer,scorer,conservative=False)
            for candidate in candidates:
                if 'metrics' not in candidate:
                    candidate['metrics']=rouge_scores(candidate['summary'],row['references'])
            row.update({'candidates':candidates,'reranked':best,'repair':repair,
                        'repair_seconds':time.perf_counter()-start})
            row['total_seconds']+=row['repair_seconds']
            values.append(row)
            if (i+1)%8==0:
                print(f'[동일 예산 재생성] {stage} {i+1}/{len(original["examples"])}',flush=True)
        arms[stage]=values
        report['aggregate'][stage+'_repaired']=aggregate_candidates(values,'reranked')
        report['aggregate'][stage+'_top1']=aggregate_candidates(unrepaired,'top1')
        report['aggregate'][stage+'_reranked']=aggregate_candidates(unrepaired,'reranked')
        report['aggregate'][stage+'_repaired']['repair_attempt_count']=sum(r['repair'] is not None for r in values)
        report['aggregate'][stage+'_repaired']['repair_selected_count']=sum('repair_evidence' in r['candidates'][r['reranked']] for r in values)
    report['examples']=[{'id':b['id'],'before':b,'after':a} for b,a in zip(arms['before'],arms['after'])]
    write_json(path,report)
    setup_tracking()
    with mlflow.start_run(run_name=f'fixed_subject_repair_32_{device}'):
        mlflow.log_params({'count':32,'device':device,'same_repair_budget':True,'max_extra_passes':1,'num_beams':4})
        for arm,metrics in report['aggregate'].items():
            mlflow.log_metrics({f'{arm}_{k}':v for k,v in metrics.items()})
        mlflow.log_artifact(str(path))
    print(json.dumps(report['aggregate'],ensure_ascii=False,indent=2))


### 반복 정리 적용 결과 비교

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def compare_service_cleanup():
    path=COMPARISON_DIRECTORY/'service_comparison.json'
    previous=json.loads((COMPARISON_DIRECTORY/'comparison_repair.json').read_text(encoding='utf-8'))
    torch.set_num_threads(4)
    scorer=SupportScorer(previous['device'])
    arms={}
    items=[]
    for stage in ['before','after']:
        rows=[]
        for pair in previous['examples']:
            row=pair[stage]
            start=time.perf_counter()
            candidate=finish_candidate(row['source'],row['candidates'][row['reranked']],scorer)
            candidate['metrics']=rouge_scores(candidate['summary'],row['references'])
            rows.append({'id':row['id'],'source':row['source'],'references':row['references'],
                'candidates':[candidate],'reranked':0,'generation_seconds':row['generation_seconds'],
                'total_seconds':row['total_seconds']+time.perf_counter()-start})
        arms[stage]=rows
    aggregates={stage+'_service':aggregate_candidates(rows,'reranked') for stage,rows in arms.items()}
    for stage,rows in arms.items():
        aggregates[stage+'_service']['cleanup_document_count']=sum(bool(r['candidates'][0]['repetition_cleanup']) for r in rows)
    report={'count':len(arms['before']),'scope':previous['scope'],'device':previous['device'],
        'same_policy_before_after':True,'repetition_cleanup':'adjacent exact duplicate words absent in source only; raw generation retained',
        'repair_comparison_sha256':sha256(COMPARISON_DIRECTORY/'comparison_repair.json'),'aggregate':aggregates,
        'examples':[{'id':b['id'],'source':b['source'],'references':b['references'],
                     'before':b['candidates'][0],'after':a['candidates'][0]} for b,a in zip(arms['before'],arms['after'])]}
    write_json(path,report)
    setup_tracking()
    with mlflow.start_run(run_name=f'final_service_policy_32_{previous["device"]}'):
        mlflow.log_params({'count':len(arms['before']),'same_policy_before_after':True,'repetition_cleanup_logged_separately':True})
        for arm,metrics in aggregates.items():
            mlflow.log_metrics({f'{arm}_{k}':v for k,v in metrics.items()})
        mlflow.log_artifact(str(path))
    print(json.dumps(aggregates,ensure_ascii=False,indent=2))


### 실제 서비스의 보수적 선택 정책 비교

아래 함수를 정의합니다. 함수 정의만으로 학습을 실행하지 않습니다.

In [ ]:
def compare_service_policy():
    path=COMPARISON_DIRECTORY/'service_comparison_v4.json'
    previous=json.loads((COMPARISON_DIRECTORY/'comparison_repair.json').read_text(encoding='utf-8'))
    release=json.loads((FACTUAL_MODEL_DIRECTORY/'release.json').read_text(encoding='utf-8'))
    torch.set_num_threads(4)
    scorer=SupportScorer(previous['device'])
    tokenizer=load_tokenizer()
    arms={}
    for stage in ['before','after']:
        model=None
        rows=[]
        for pair in previous['examples']:
            row=pair[stage]
            candidates=copy.deepcopy(row['candidates'][:4])
            best=choose_candidate(row['evidence'],candidates)
            first=candidates[0]
            risk=(multiple_subject_risk(row['evidence'],first['summary']) or punctuation_repetition(first['summary'])
                  or first['repeated_word_trigrams'] or first['numbers_not_in_evidence'])
            need_repair=risk and not any(c['support']>=.5 and not punctuation_repetition(c['summary'])
                and not c['repeated_word_trigrams'] and not c['numbers_not_in_evidence'] for c in candidates)
            repair=None
            if need_repair:
                desired_premise,_=anchored_premise(row['source'],candidates[best]['summary'])
                if row['repair'] and row['repair']['premise']==desired_premise:
                    candidates=copy.deepcopy(row['candidates'])
                    best=choose_candidate(row['evidence'],candidates)
                    repair=row['repair']
                else:
                    if model is None:
                        model=load_rewriter().to(previous['device'])
                        if stage=='after':
                            model.load_state_dict(load_file(str(FACTUAL_MODEL_DIRECTORY/release['checkpoint'])),strict=False)
                    candidates,best,repair=repair_candidates(row['source'],row['evidence'],candidates,best,model,tokenizer,scorer)
            final=finish_candidate(row['source'],candidates[best],scorer)
            final['metrics']=rouge_scores(final['summary'],row['references'])
            rows.append({'id':row['id'],'source':row['source'],'references':row['references'],
                'candidates':[final],'reranked':0,'generation_seconds':row['generation_seconds'],
                'total_seconds':row['total_seconds'],'repair':repair})
        arms[stage]=rows
        del model
    aggregates={stage+'_service':aggregate_candidates(rows,'reranked') for stage,rows in arms.items()}
    for stage,rows in arms.items():
        metrics=aggregates[stage+'_service']
        metrics.pop('median_seconds')
        metrics['punctuation_repetition_count']=sum(bool(punctuation_repetition(r['candidates'][0]['summary'])) for r in rows)
        metrics['cleanup_document_count']=sum(bool(r['candidates'][0]['repetition_cleanup']) for r in rows)
        metrics['repair_attempt_count']=sum(r['repair'] is not None for r in rows)
        metrics['repair_selected_count']=sum('repair_evidence' in r['candidates'][0] for r in rows)
    report={'count':len(arms['before']),'scope':'development revision after inspection of earlier 32-document results; not held-out testing',
        'device':previous['device'],'same_policy_before_after':True,
        'policy_version':'conservative_subject_context_v4',
        'policy':'preserve original top candidate unless explicit multiple-subject risk, repeated words, or unsupported numbers; restore subject context from original source; same maximum one extra generation pass',
        'timing_scope':'cached generation/scoring outputs reused; not a new latency benchmark',
        'repair_comparison_sha256':sha256(COMPARISON_DIRECTORY/'comparison_repair.json'),'aggregate':aggregates,
        'examples':[{'id':b['id'],'source':b['source'],'references':b['references'],
                     'before':b['candidates'][0],'after':a['candidates'][0]} for b,a in zip(arms['before'],arms['after'])]}
    write_json(path,report)
    setup_tracking()
    with mlflow.start_run(run_name=f'conservative_service_v4_32_{previous["device"]}'):
        mlflow.log_params({'count':len(arms['before']),'same_policy_before_after':True,'version':'v4','evaluation_role':'development_not_test'})
        for arm,metrics in aggregates.items():
            mlflow.log_metrics({f'{arm}_{k}':v for k,v in metrics.items()})
        mlflow.log_artifact(str(path))
    print(json.dumps(aggregates,ensure_ascii=False,indent=2))


## 자동 실행

모든 함수 준비가 끝났습니다. 아래 셀은 위에서부터 의존 순서대로 실행되며 같은 조건의 완료 모델은 건너뜁니다.

### 데이터·토큰화 자동 준비

원본·전처리 설정·코드가 같으면 저장 자료를 재사용합니다. 변경 시 새 경로에서 다시 준비합니다.

In [ ]:
raw_inputs = {spec["name"]: file_fingerprint(DATA_DIR / spec["name"]) for spec in source_info["full_archives"]}
data_step = cache_context("preprocess", {"seed": DATA_SEED, "scope": sorted(SCOPE), "domain_terms": DOMAIN_TERMS},
                          {"archives": raw_inputs, "source_info": file_fingerprint(DATA_DIR / "source_info.json")},
                          [clean_text, digest, document_key, connected_group_ids, assign_splits,
                           separation_checks, original_overlap, prepare_preprocessing],
                          {**{split: f"news_briefing_{split}.jsonl" for split in SPLITS},
                           "report": "news_briefing_preprocess_report.json"})
DATA_OUTPUTS = {split: data_step["paths"][split] for split in SPLITS}
PREPROCESS_REPORT = data_step["paths"]["report"]
finish_step(data_step, prepare_preprocessing)

token_step = cache_context("tokenization", TOKEN_SETTINGS,
                           {"data": data_step["fingerprint"],
                            "datasets": {split: file_fingerprint(DATA_OUTPUTS[split]) for split in ["train", "validation"]},
                            "tokenizer": file_fingerprint(TOKENIZER_DIR / "tokenizer.json"),
                            "tokenizer_config": {path.name: file_fingerprint(path) for path in TOKENIZER_DIR.glob("*.json")}},
                           [normalize_text, input_ids, load_tokenizer, read_training_rows, tokenize_records, prepare_tokenization],
                           {"preparation": "kobart_training_preparation.json",
                            **{split: f"news_briefing_{split}_tokens.npz" for split in ["train", "validation"]}})
TOKEN_REPORT_PATH = token_step["paths"]["preparation"]
TOKEN_FILES = {split: token_step["paths"][split] for split in ["train", "validation"]}
finish_step(token_step, prepare_tokenization)


### 시험·주제 모델 자동 학습

학습률·epoch·배치·표본·생성 설정·시작 가중치·관련 함수가 같으면 학습을 건너뜁니다. 시험 모델이 바뀌면 주제 모델도 새로 학습합니다.

In [ ]:
BASE_DIRECTORIES, BASE_RESULTS, BASE_STEPS = {}, {}, {}
base_functions = [normalize_text, input_ids, load_model, load_tokenizer, generate,
                  number_tokens, char_lcs_f1, sentence_candidates, review_signals,
                  read_training_rows, stratified_indices, training_config, training_samples, domain_pool,
                  token_feature, token_batch, model_loss, validation_predictions, fit_one_epoch, train_kobart]
for stage_name in ["pilot", "domain"]:
    initial = TOKENIZER_DIR if stage_name == "pilot" else BASE_DIRECTORIES["pilot"]
    inputs = {"tokens": token_step["fingerprint"],
              "token_files": {split: file_fingerprint(path) for split, path in TOKEN_FILES.items()},
              "initial_weights": file_fingerprint(initial / "model.safetensors"),
              "model_config": file_fingerprint(initial / "config.json")}
    if stage_name == "domain":
        inputs.update(pilot=BASE_STEPS["pilot"]["fingerprint"], plan=file_fingerprint(DOMAIN_PLAN_PATH))
    context = cache_context(stage_name, {"training": BASE_SETTINGS[stage_name], "generation": BASE_GENERATION},
                            inputs, base_functions,
                            {"weights": "model.safetensors", "config": "config.json",
                             "generation": "generation_config.json", "results": f"{stage_name}_results.json"})
    BASE_DIRECTORIES[stage_name] = context["directory"]
    finish_step(context, lambda: train_kobart(stage_name, context["directory"]))
    BASE_STEPS[stage_name] = context
    BASE_RESULTS[stage_name] = read_json(context["paths"]["results"])

DOMAIN_DIRECTORY = BASE_DIRECTORIES["domain"]
original_release = read_json(ROOT / "artifacts/model_release.json")
ACTIVE_BASE_RELEASE = {"model_directory": DOMAIN_DIRECTORY.relative_to(ROOT).as_posix(),
                       "weights_sha256": file_fingerprint(DOMAIN_DIRECTORY / "model.safetensors"),
                       "generation_settings": BASE_GENERATION,
                       "max_source_tokens": read_json(DOMAIN_PLAN_PATH)["rules"]["max_source_tokens"]}
BASE_RELEASE_FINGERPRINT = (file_fingerprint(ROOT / "artifacts/model_release.json")
                            if all(ACTIVE_BASE_RELEASE[key] == original_release[key] for key in ACTIVE_BASE_RELEASE)
                            else json_fingerprint(ACTIVE_BASE_RELEASE))


### 문장 선별기·재작성 생성기 자동 학습

선별기 구조·학습 설정·입력 모델 변경을 감지합니다. 선별기가 바뀌면 생성기의 학습도 다시 진행합니다.

In [ ]:
rewrite_functions = [normalize_text, input_ids, document_features, sentence_units, selected_rows, read_release,
                     SentenceSelector.__init__, SentenceSelector.forward, select_evidence, rewriting_signals,
                     overlap_label, load_model, load_selector, load_rewriter, decoder_parameter]
selector_step = cache_context("selector", {"training": SELECTOR_SETTINGS, "generation": GENERATION,
                                          "sample_seed": REWRITE_SEED,
                                          "document_tokens": MAX_DOCUMENT_TOKENS, "evidence_tokens": MAX_EVIDENCE_TOKENS},
                              {"base": BASE_STEPS["domain"]["fingerprint"], "data": data_step["fingerprint"],
                               "base_release": BASE_RELEASE_FINGERPRINT},
                              rewrite_functions + [train_sentence_selector],
                              {"weights": "rewrite_selector.pt", "manifest": "rewrite_release.json"})
SELECTOR, MANIFEST = selector_step["paths"]["weights"], selector_step["paths"]["manifest"]
finish_step(selector_step, train_sentence_selector)

generator_step = cache_context("generator", {"training": GENERATOR_SETTINGS, "selection": SELECTOR_SETTINGS,
                                             "sample_seed": REWRITE_SEED,
                                             "generation": GENERATION, "document_tokens": MAX_DOCUMENT_TOKENS,
                                             "evidence_tokens": MAX_EVIDENCE_TOKENS},
                               {"selector": selector_step["fingerprint"], "base": BASE_STEPS["domain"]["fingerprint"],
                                "selector_weights": file_fingerprint(SELECTOR), "selector_manifest": file_fingerprint(MANIFEST)},
                               rewrite_functions + [train_rewriter_generator, review_signals, number_tokens, char_lcs_f1],
                               {"weights": "rewrite_decoder.safetensors", "manifest": "rewrite_generator.json"})
DECODER, GENERATOR_MANIFEST = generator_step["paths"]["weights"], generator_step["paths"]["manifest"]
finish_step(generator_step, train_rewriter_generator)


### 추가 학습 자료와 학습률 실험 자동 실행

반복 벌점·학습률·epoch·학습할 층·근거 선택 정책을 비교합니다. 앞 단계나 데이터가 바뀌면 다시 학습합니다.

In [ ]:
factual_data_step = cache_context("factual_data", {"selection": SELECTOR_SETTINGS, "generator": GENERATOR_SETTINGS,
                                                  "sample_seed": REWRITE_SEED,
                                                  "tuning_count": FACTUAL_SETTINGS["tuning_count"],
                                                  "development_count": FACTUAL_SETTINGS["development_count"]},
                                  {"generator": generator_step["fingerprint"], "selector": selector_step["fingerprint"],
                                   "generator_weights": file_fingerprint(DECODER), "data": data_step["fingerprint"]},
                                  rewrite_functions + [prepare_factual_data, review_signals, number_tokens, char_lcs_f1],
                                  {"data": "data.json"})
FACTUAL_DATA_FILE = factual_data_step["paths"]["data"]
finish_step(factual_data_step, lambda: prepare_factual_data(DEVICE))

factual_functions = [factual_parameter, set_factual_seed, decoder_state, build_cache, output_loss,
                     validation_loss, evaluate_rows, aggregate_candidates, train_factual_grid,
                     generate_candidates, rank_candidates, choose_candidate, subject_anchor, anchored_premise,
                     adjacent_words, punctuation_repetition, multiple_subject_risk, review_signals, rewriting_signals,
                     rouge_scores, SupportScorer.__init__, SupportScorer._score_windows, SupportScorer.score]
factual_step = cache_context("factual", {"training": FACTUAL_SETTINGS, "generation": CANDIDATE_SETTINGS,
                                       "seed": FACTUAL_SEED, "learning_rates": LEARNING_RATES, "epochs": EPOCHS},
                             {"data": factual_data_step["fingerprint"], "data_file": file_fingerprint(FACTUAL_DATA_FILE),
                              "generator": generator_step["fingerprint"],
                              "nli": {path.name: file_fingerprint(path) for path in NLI_DIRECTORY.iterdir() if path.is_file()}},
                             factual_functions,
                             {"grid": "grid.json", "release": "release.json", "protocol": "protocol.json",
                              **{f"weights_{lr:g}_{epoch}": f"lr_{lr:g}_epoch_{epoch}.safetensors"
                                 for lr in LEARNING_RATES for epoch in range(1, EPOCHS + 1)}})
FACTUAL_DIRECTORY = FACTUAL_MODEL_DIRECTORY = factual_step["directory"]
finish_step(factual_step, lambda: train_factual_grid(DEVICE))


### 현재 모델의 검증·개발 비교 자동 실행

평가도 모델·데이터·생성 설정이 같은 경우에만 재사용합니다. 새 모델은 검증·개발 자료로 비교하며 기존 서비스 선택과 최종 test 기록은 보존합니다.

In [ ]:
validation_step = cache_context("domain_validation", BASE_GENERATION,
                                {"domain": BASE_STEPS["domain"]["fingerprint"],
                                 "results": file_fingerprint(DOMAIN_DIRECTORY / "domain_results.json")},
                                [aggregate_metrics, evaluate_validation, rouge_scores, review_signals],
                                {"report": "validation_evaluation.json"})
VALIDATION_REPORT_PATH = validation_step["paths"]["report"]
finish_step(validation_step, evaluate_validation)

rewrite_step = cache_context("rewrite_validation", {"generation": GENERATION,
                                                    "tuning_count": FACTUAL_SETTINGS["tuning_count"],
                                                    "development_count": FACTUAL_SETTINGS["development_count"]},
                             {"generator": generator_step["fingerprint"], "selector": selector_step["fingerprint"]},
                             rewrite_functions + [evaluate_rewriter, summarize_document, rouge_scores, generate],
                             {"report": "rewrite_validation_v2.json"})
REWRITE_REPORT_PATH = rewrite_step["paths"]["report"]
finish_step(rewrite_step, evaluate_rewriter)

comparison_functions = factual_functions + [compare_factual_models, compare_repairs, compare_service_cleanup,
                                            compare_service_policy, repair_candidates, finish_candidate,
                                            clean_accidental_repetition, load_rewriter]
comparison_step = cache_context("factual_comparison", {"generation": CANDIDATE_SETTINGS, "training": FACTUAL_SETTINGS},
                                {"model": factual_step["fingerprint"], "generator": generator_step["fingerprint"],
                                 "data": file_fingerprint(FACTUAL_DATA_FILE)}, comparison_functions,
                                {"comparison": "comparison.json", "repair": "comparison_repair.json",
                                 "cleanup": "service_comparison.json", "service": "service_comparison_v4.json"})
COMPARISON_DIRECTORY = comparison_step["directory"]

def build_comparisons():
    compare_factual_models(DEVICE)
    compare_repairs()
    compare_service_cleanup()
    compare_service_policy()

finish_step(comparison_step, build_comparisons)
write_json(FILE_HASH_PATH, FILE_HASHES)
display(pd.DataFrame([{"단계": name, "학습·처리": "저장 결과 재사용" if context["cached"] else "새로 실행",
                       "설정 식별값": context["fingerprint"][:12]}
                      for name, context in TRAINING_STEPS.items()]))


## 8. 완료된 결과 확인

### 시험·주제 학습 곡선 확인

저장된 손실 기록만 읽습니다. 이 셀은 모델을 추가 학습하지 않습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for stage, ax in zip(["pilot", "domain"], axes):
    path = BASE_DIRECTORIES[stage] / f"{stage}_results.json"
    if path.exists():
        history = pd.DataFrame(read_json(path)["history"])
        ax.plot(history["update"], history["train_monitor_loss"], label="train monitor")
        ax.plot(history["update"], history["validation_loss"], label="validation")
        ax.legend()
    ax.set(title=stage, xlabel="optimizer update", ylabel="token loss")
plt.tight_layout()
plt.show()


### 학습률·epoch 결과와 서비스 채택 상태 확인

검증 손실로 고른 실험 가중치와 실제 화면에서 사용하는 가중치를 구분해 확인합니다.

In [ ]:
grid_path = FACTUAL_DIRECTORY / "grid.json"
if grid_path.exists():
    results = read_json(grid_path)
    table = pd.DataFrame(results["epochs"])[["learning_rate", "epoch", "train_loss", "validation_loss"]]
    display(table)
    for learning_rate, group in table.groupby("learning_rate"):
        plt.plot(group["epoch"], group["validation_loss"], marker="o", label=f"lr={learning_rate:g}")
    plt.xlabel("epoch")
    plt.ylabel("validation loss")
    plt.legend()
    plt.show()
if (ROOT / "artifacts/factual/service_release.json").exists():
    service = read_json(ROOT / "artifacts/factual/service_release.json")
    print("서비스 상태:", service["status"])
    print("기존 생성기 사용:", service.get("use_initial_model", False))
    print("추가 학습 가중치 채택:", service.get("trained_candidate", {}).get("adopted", False))


### 검증·최종 평가와 개발 비교 확인

최종 test와 개발 비교는 다른 자료입니다. 점수는 F1×100이며 사실 정확도 비율이 아닙니다.

In [ ]:
validation = read_json(VALIDATION_REPORT_PATH)
print("현재 학습 모델의 검증 결과:", validation["n"])
display((pd.DataFrame(validation["metrics"]).T[["rouge1", "rouge2", "rougeL", "char_lcs_f1"]] * 100).round(2))
final_test_path = ROOT / "artifacts/kobart_domain/test_evaluation.json"
if final_test_path.exists():
    result = read_json(final_test_path)
    print("기존 고정 서비스 모델의 최종 test 기록:", result["n"])
    display((pd.DataFrame(result["metrics"]).T[["rouge1", "rouge2", "rougeL", "char_lcs_f1"]] * 100).round(2))
comparison = read_json(COMPARISON_DIRECTORY / "service_comparison_v4.json")
print("현재 모델의 서비스 정책 개발 비교:", comparison["scope"])
display(pd.DataFrame(comparison["aggregate"]).T)
example = comparison["examples"][0]
print("원문:", example["source"])
print("기존:", example["before"]["summary"])
print("추가 학습:", example["after"]["summary"])


## 화면과 실험 기록 실행

프로젝트 터미널에서 화면을 실행합니다.

```powershell
.\.venv\Scripts\python.exe -m streamlit run .\app.py
```

MLflow 기록은 다음 명령으로 확인합니다.

```powershell
.\.venv\Scripts\python.exe -m mlflow server --backend-store-uri sqlite:///artifacts/factual/mlflow.db --host 127.0.0.1 --port 5001 --workers 1
```

데이터: 과학기술정보통신부·한국지능정보사회진흥원 AI Hub의 ‘요약문 및 레포트 생성 데이터’(582).
이 노트북의 전처리·학습·평가는 로컬 파일로 수행합니다. PDF·DOCX·TXT 입력 처리는 `app.py`에 있습니다.